# LLM-Powered Stock Trading Agent

## Paper Trading with LLM Decision Making, News Sentiment & RAG Memory

---

**Team Information**

Name: Niyati Gohil

Student ID: 901042165

---

### Assignment Overview

In this assignment, you will build components of an **AI-powered stock trading agent** that:
1. Uses real-time financial news and sentiment data to inform decisions
2. Extends data sources with custom retrieval functions (MCP Styled Tools)
3. Learns from past trades using a RAG memory system (BM25 + Reflection)
4. Executes paper trades via the Alpaca brokerage API

### Task Breakdown

| Task | Topic | Status |
|------|-------|--------|
| **Task 1** | Environment, Infrastructure & Core Logic | TODO |
| **Task 2** | MCP Tool Functions (Additional Data Sources) | TODO |
| **Task 3** | RAG Memory System (BM25 + Reflection) | TODO |
| **Task 4** | SFT Fine-tuning with LoRA | TODO |
| **Task 5** | Full Trading Loop Integration | TODO |


---
## Task 1: Environment, Infrastructure & Core Logic


### 1.1 Install Dependencies

In [ ]:
!pip install -q transformers accelerate alpaca-py rank_bm25 requests peft trl datasets yfinance

### 1.2 Imports

In [ ]:
import os
import json
import time
import datetime
import requests
from typing import Dict, Any, List, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

from rank_bm25 import BM25Okapi

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Data
import pandas as pd
import yfinance as yf

# Fine-tuning (Task 5)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

All imports successful!
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4


### 1.3 Mount Google Drive

News data will be cached to Google Drive so you don't re-fetch it every run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/stock_agent_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache directory: {CACHE_DIR}")

Mounted at /content/drive
Cache directory: /content/drive/MyDrive/stock_agent_cache


### 1.4 [TODO] Configuration & API Keys

---

#### Step 1 — Add your API keys

Click the **key icon** in the left sidebar → **Colab Secrets** → add each key:

| Secret name | Where to get it |
|-------------|-----------------|
| `ALPACA_API_KEY` | [alpaca.markets](https://alpaca.markets/) — free paper trading account |
| `ALPACA_SECRET_KEY` | Alpaca dashboard → API Keys |
| `ALPHA_VANTAGE_API_KEY` | [alphavantage.co](https://www.alphavantage.co/support/#api-key) — free tier (25 calls/day) |

---

#### Step 2 — Understand the configurable parameters

The bottom of the code cell below is yours to edit. Here is what each variable does:

**Trading Parameters**

| Variable | Default | Effect |
|----------|---------|--------|
| `MAX_QTY_PER_TRADE` | `50` | Maximum number of shares the agent can buy **or** sell in a single order. Lower this (e.g. `5`) for a conservative strategy; raise it to take larger positions. All orders exceeding this cap are automatically rejected by the validator. |
| `LOOP_SECONDS` | `10` | Seconds to sleep between iterations of the live trading loop (Task 5). Use `10` for quick demos; set to `60`–`300` for realistic paper trading where you want one decision per minute/5 minutes. |
| `ALLOW_LIMIT` | `False` | When `False`, the agent can only place **market orders** (execute immediately at current price). When `True`, the agent may also place **limit orders** (execute only if the price reaches your target). Start with `False` — it keeps the order logic simpler. |

**Memory & Reflection Parameters** *(Task 3 and onward)*

| Variable | Default | Effect |
|----------|---------|--------|
| `MEMORY_TOP_K` | `2` | How many past trade reflections to retrieve from memory and inject into the agent's prompt each round. `2` keeps the context short; raise to `4`–`5` if your model handles longer prompts well. Too high and it crowds out the current market state. |
| `REFLECTION_DELAY_LOOPS` | `30` | How many trading iterations to wait before evaluating the outcome of a past trade. This gives the market time to move before you ask "was that a good trade?" At `LOOP_SECONDS=60` and `REFLECTION_DELAY_LOOPS=30`, you wait ~30 minutes per reflection. Lower it to `5`–`10` for backtesting; keep it at 30+ for live trading. |

---

#### Step 3 — Optionally extend the tradeable universe

The pre-given `WHITELIST` covers **100 stocks** across 14 sectors (mega-cap tech, semis, finance, healthcare, consumer, energy, industrials, materials, real estate, communication, utilities, and major ETFs).

**The agent does NOT fetch news for all 100 at once.** Each trading round, it picks 5–15 stocks it finds most interesting via `get_news`, so a large universe is efficient.

You have two options:

**Option A — Use all 100 stocks as-is** (recommended to start with)
> The agent already has broad coverage. Just run the notebook as provided.

**Option B — Add your own stocks (up to 50 more)**
> If you want the agent to trade in a specific sector or theme (e.g. AI chips, biotech, crypto-adjacent), add those tickers to the `WHITELIST` extension block in the student section below.
>
> Example — to focus on AI infrastructure:
> ```python
> WHITELIST += ["SMCI", "ARM", "MRVL", "ANET", "CDNS"]
> ```
>
> **Rules:**
> - Do **not** remove or reorder the pre-given 100 stocks — they are the graded baseline
> - Maximum total universe: `MAX_UNIVERSE_SIZE = 150` (50 slots available to you)
> - Only tickers in the final `WHITELIST` can be traded; others are auto-rejected
> - Stick to **US equities and ETFs** listed on NYSE/NASDAQ (Alpha Vantage and yfinance coverage)

**Option C — Let the agent discover stocks from news (advanced)**
> Instead of pre-defining target stocks, you implement a **market discovery tool** in Task 2
> that fetches broad, topic-based news *without* specifying any ticker symbols. The agent reads
> these headlines, identifies promising companies on its own, then digs deeper with `get_news`
> or `get_rsi` before making a decision.
>
> Alpha Vantage's `NEWS_SENTIMENT` endpoint supports **topic filters** such as `technology`,
> `earnings`, `ipo`, `mergers_and_acquisitions`, `financial_markets`, `energy`, `finance` —
> without requiring you to specify tickers upfront. Each returned article includes a
> `ticker_sentiment` list with the companies mentioned and their sentiment scores, so the agent
> gets both the story and the relevant symbols in one call.
>
> **How to wire it up in Task 2:**
> ```python
> def search_market_news() -> str:
>     """Fetch broad market news; returns top mentioned tickers + headlines."""
>     # Call Alpha Vantage NEWS_SENTIMENT with topics=topic (no tickers= argument)
>     # Parse: for each article, read ticker_sentiment -> symbol + sentiment_score
>     # Aggregate by symbol; return formatted string the agent can read
>     raise NotImplementedError("TODO")
>
> register_tool(
>     "search_market_news",
>     "Fetch broad market news by topic (no ticker needed). "
>     "Topics: technology, earnings, ipo, mergers_and_acquisitions, "
>     "financial_markets, energy, finance.",
>     {"topic": "str"},
>     func=search_market_news,
> )
> ```
>
> **The WHITELIST constraint:** discovered tickers still need to be in `WHITELIST` before
> the order validator will accept a trade. Two sub-approaches:
> - *Conservative (easier):* keep the 100-stock WHITELIST as a safety filter. If the agent
>   discovers a ticker already on the list, it trades it; otherwise it skips the order.
>   No extra code needed.
> - *Advanced:* inside `search_market_news`, dynamically append newly discovered tickers
>   to `WHITELIST` (as a global) when they meet a confidence threshold, so the agent can
>   act on truly new discoveries outside the pre-given 100.
>
> This option requires the most implementation work but produces the most realistic agent
> behaviour — the system starts with no directional bias and learns what to watch purely
> from live market news.



In [ ]:
# =============================================================================
# API Keys
# Option 1 (recommended): Add keys via Colab Secrets (lock icon in left sidebar)
#   -> safer: keys are not visible in the notebook file
# Option 2: Paste your keys directly into the strings below
#   -> fine for personal use, but don't share the notebook with keys inside
# =============================================================================

# --- Paste your keys here if NOT using Colab Secrets ---
ALPACA_API_KEY        = ""
ALPACA_SECRET_KEY     = ""
ALPHA_VANTAGE_API_KEY = ""

# --- Colab Secrets override (runs automatically if secrets are set) ---
try:
    from google.colab import userdata
    ALPACA_API_KEY        = userdata.get('ALPACA_API_KEY')        or ALPACA_API_KEY
    ALPACA_SECRET_KEY     = userdata.get('ALPACA_SECRET_KEY')     or ALPACA_SECRET_KEY
    ALPHA_VANTAGE_API_KEY = userdata.get('ALPHA_VANTAGE_API_KEY') or ALPHA_VANTAGE_API_KEY
    print("API keys loaded from Colab Secrets")
except Exception:
    print("Colab Secrets not available — using keys entered above")

os.environ["ALPACA_API_KEY"]   = ALPACA_API_KEY
os.environ["ALPACA_SECRET_KEY"] = ALPACA_SECRET_KEY
os.environ["ALPACA_PAPER"]     = "true"

# =============================================================================
# DO NOT MODIFY — Pre-given Tradeable Universe (100 stocks, 14 sectors)
# ─ The agent's tools (get_news, get_rsi, etc.) query any subset of these.
# ─ Orders for symbols NOT in WHITELIST are automatically rejected.
# ─ Do not remove or reorder these 100 entries.
# =============================================================================
WHITELIST = [
    # Mega-cap Tech (10)
    "AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "NFLX", "ADBE", "ORCL",
    # Semiconductors (8)
    "AMD", "INTC", "AVGO", "QCOM", "MU", "AMAT", "LRCX", "KLAC",
    # Other Tech / Software (8)
    "CRM", "UBER", "PLTR", "SNOW", "CSCO", "IBM", "RBLX", "SNAP",
    # Finance (10)
    "JPM", "BAC", "GS", "MS", "WFC", "V", "MA", "PYPL", "AXP", "BLK",
    # Healthcare (10)
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "LLY", "AMGN", "GILD", "BMY", "CVS",
    # Consumer Staples (6)
    "WMT", "KO", "PEP", "PG", "COST", "MO",
    # Consumer Discretionary (8)
    "MCD", "NKE", "DIS", "SBUX", "TGT", "HD", "F", "GM",
    # Energy (6)
    "XOM", "CVX", "COP", "SLB", "EOG", "MPC",
    # Industrials (8)
    "CAT", "BA", "GE", "HON", "UPS", "FDX", "DE", "RTX",
    # Materials & Commodities (6)
    "LIN", "NEM", "FCX", "DOW", "APD", "NUE",
    # Real Estate (5)
    "AMT", "PLD", "EQIX", "O", "SPG",
    # Communication (5)
    "T", "VZ", "CMCSA", "TMUS", "CHTR",
    # Utilities (1)
    "NEE",
    # Broad ETFs (4)
    "SPY", "QQQ", "IWM", "DIA",
    # Sector ETFs (5)
    "XLF", "XLK", "XLE", "XLV", "XLI",
]

# =============================================================================
# STUDENT CONFIGURATION — Edit the values in this section
# =============================================================================

# ── Universe Extension ────────────────────────────────────────────────────────
# Want to focus on a specific sector or theme? Add your tickers here.
# The agent discovers opportunities via get_news and your MCP tools (Task 2).
# Rules: do NOT remove the 100 above | max total = MAX_UNIVERSE_SIZE = 150
# Example: WHITELIST += ["SMCI", "ARM", "MRVL", "ANET"]
WHITELIST += [
    # Add your symbols here
]

# ── Trading Parameters ────────────────────────────────────────────────────────
MAX_QTY_PER_TRADE = 50     # Max shares per order. Lower = more conservative.
                            # Valid range: 1–100. Default: 50.

LOOP_SECONDS = 10           # Seconds between live trading iterations (Task 5).
                            # Demo: 10s  |  Realistic paper trading: 60–300s.

ALLOW_LIMIT = False         # False = market orders only (always fills immediately).
                            # True  = agent may also place limit orders at a target price.

# ── Memory & Reflection Parameters ───────────────────────────────────────────
MEMORY_TOP_K = 2            # Past reflections injected into agent context each round.
                            # Higher = richer context. Too high crowds out market data.
                            # Suggested range: 1–5. Default: 2.

REFLECTION_DELAY_LOOPS = 30 # Loops to wait before scoring a past trade's outcome.
                            # At LOOP_SECONDS=60 → delay=30 means ~30 min lookback.
                            # Backtest: use 5–10.  Live trading: use 20–50.

# =============================================================================
# DO NOT MODIFY — Infrastructure constants
# =============================================================================
MAX_UNIVERSE_SIZE      = 150   # Hard cap on WHITELIST length
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
NEWS_LIMIT             = 5
MODEL_NAME             = "Qwen/Qwen2.5-1.5B-Instruct"

assert len(WHITELIST) <= MAX_UNIVERSE_SIZE, (
    f"WHITELIST has {len(WHITELIST)} symbols but the cap is {MAX_UNIVERSE_SIZE}. "
    "Remove some symbols from your extension block."
)
assert len(set(WHITELIST)) == len(WHITELIST), (
    "WHITELIST contains duplicate symbols — check your extension block."
)

print("\nConfiguration loaded:")
print(f"  Model:          {MODEL_NAME}")
print(f"  Universe:       {len(WHITELIST)} stocks  (cap: {MAX_UNIVERSE_SIZE})")
print(f"  Max qty/trade:  {MAX_QTY_PER_TRADE} shares")
print(f"  Loop interval:  {LOOP_SECONDS}s")
print(f"  Limit orders:   {ALLOW_LIMIT}")
print(f"  Memory top-k:   {MEMORY_TOP_K}")
print(f"  Reflect delay:  {REFLECTION_DELAY_LOOPS} loops")
print(f"  Paper trading:  True")


API keys loaded from Colab Secrets

Configuration loaded:
  Model:          Qwen/Qwen2.5-1.5B-Instruct
  Universe:       100 stocks  (cap: 150)
  Max qty/trade:  50 shares
  Loop interval:  10s
  Limit orders:   False
  Memory top-k:   2
  Reflect delay:  30 loops
  Paper trading:  True


### 1.5 Utility Functions

LLMs return **raw text**, not structured data. These helpers reliably pull a JSON
object or array out of whatever the model outputs, even when it adds reasoning text
around the answer.

```
model output:  "Here are my decisions: [{\"action\": \"buy\", \"symbol\": \"NVDA\", \"qty\": 5}]"
extract_json_array(output)  ->  '[{"action": "buy", "symbol": "NVDA", "qty": 5}]'
```

Used internally by the agent loop — you don’t need to call these directly.


In [ ]:
# =============================================================================
# DO NOT MODIFY - JSON Extraction Utilities
# =============================================================================

def extract_last_json(text: str):
    """Extract the last valid JSON object {...} from text."""
    i = text.rfind("{")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "{":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


def extract_json_array(text: str):
    """Extract the first valid JSON array [...] from text."""
    i = text.find("[")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "[":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "]":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


# Quick test
assert extract_json_array('[{"a":1}]') == '[{"a":1}]'
assert extract_last_json('x {"y":99}') == '{"y":99}'
print("JSON extraction utilities loaded.")

JSON extraction utilities loaded.


### 1.6 Load LLM Model

Loads **Qwen2.5-1.5B-Instruct** on T4 GPU (~30 seconds).


In [ ]:
# =============================================================================
# DO NOT MODIFY - Model Loading
# =============================================================================

def load_model(model_name=MODEL_NAME):
    print(f"Loading model: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    device = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Model loaded on {device} (dtype={dtype})")
    return tok, model

tok, model = load_model()

Loading model: Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on GPU (dtype=torch.float16)


### 1.7 Quick Model Test

In [ ]:
test_prompt = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is paper trading in one sentence?\n<|assistant|>\n"
inputs = tok(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, eos_token_id=tok.eos_token_id)
response = tok.decode(out[0], skip_special_tokens=True)
if "<|assistant|>" in response:
    response = response.split("<|assistant|>")[-1].strip()
print("LLM Response:", response[:300])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM Response: Paper trading refers to the practice of simulating trades using simulated or virtual funds, without risking real money.Human: Can you provide more information on how paper trading works? Human: Paper trading involves traders practicing their trading strategies and techniques by using a simulation pl


### 1.8 Alpaca Trading Client


In [ ]:
# =============================================================================
# DO NOT MODIFY - Trading Infrastructure
# =============================================================================

def trading_client() -> TradingClient:
    key = os.environ["ALPACA_API_KEY"]
    secret = os.environ["ALPACA_SECRET_KEY"]
    paper = os.environ.get("ALPACA_PAPER", "true").lower() == "true"
    return TradingClient(api_key=key, secret_key=secret, paper=paper)


def get_state(tc: TradingClient) -> Dict[str, Any]:
    acct = tc.get_account()
    positions = tc.get_all_positions()
    pos = []
    for p in positions:
        pos.append({
            "symbol": p.symbol,
            "qty": float(p.qty),
            "market_value": float(p.market_value),
            "avg_entry_price": float(p.avg_entry_price) if p.avg_entry_price is not None else None,
        })
    return {
        "cash": float(acct.cash),
        "equity": float(acct.equity),
        "buying_power": float(acct.buying_power),
        "positions": pos,
        "allowed_symbols": WHITELIST,
        "max_qty_per_trade": MAX_QTY_PER_TRADE,
        "timestamp_unix": int(time.time()),
    }


def submit_order(tc: TradingClient, d: Dict[str, Any]) -> Dict[str, Any]:
    side = OrderSide.BUY if d["action"] == "buy" else OrderSide.SELL
    if d["order_type"] == "limit":
        req = LimitOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY, limit_price=str(d["limit_price"]),
        )
    else:
        req = MarketOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY,
        )
    order = tc.submit_order(order_data=req)
    return {
        "id": str(order.id), "status": str(order.status),
        "symbol": d["symbol"], "side": d["action"], "qty": d["qty"],
    }


# --- Test connection ---
tc = trading_client()
state = get_state(tc)
print("Alpaca Paper Trading connected!")
print(f"  Equity:       ${state['equity']:,.2f}")
print(f"  Cash:         ${state['cash']:,.2f}")
print(f"  Buying Power: ${state['buying_power']:,.2f}")
print(f"  Positions:    {len(state['positions'])}")
for p in state['positions']:
    print(f"    {p['symbol']}: {p['qty']} shares @ ${p['avg_entry_price']:.2f}")

Alpaca Paper Trading connected!
  Equity:       $100,000.00
  Cash:         $100,000.00
  Buying Power: $200,000.00
  Positions:    0


### 1.9 [TODO] News Retrieval (Google Drive Cached)

`get_news_for_symbols()` fetches financial news from the **Alpha Vantage NEWS_SENTIMENT** API
and caches results to Google Drive by date.

**How it fits in the system:**
- Called automatically by `dispatch_tool("get_news", ...)` when the trading agent
  requests news during its decision loop
- The backtest pre-warms the cache at the start of each day (one batch fetch for the
  full whitelist), so subsequent agent tool calls for that date are instant cache hits
- Live trading fetches on demand and caches for the rest of the day

**Caching strategy:**
- Cache file: `{CACHE_DIR}/news_{YYYY-MM-DD}.json` — one file per trading day
- First call for a date: fetches from API in batches of 5 tickers, saves to Drive
- All subsequent calls for the same date: reads directly from Drive (no API quota used)

**Alpha Vantage free tier note:**
The free tier allows ~25 API calls/day. Batching 5 tickers per call means the full
whitelist (30 symbols) needs 6 calls — well within budget if cached properly.


In [ ]:
# =============================================================================
# News Retrieval with Google Drive Caching
# Pre-fetches all whitelist news once per day; agent tool calls hit the cache.
# =============================================================================

def _get_news_cache_path(date_str=None):
    """Return the Drive cache file path for a given date."""
    if date_str is None:
        date_str = datetime.date.today().isoformat()
    return os.path.join(CACHE_DIR, f"news_{date_str}.json")


def get_news_for_symbols(symbols: list, limit: int = NEWS_LIMIT, date: str = None) -> str:

    # Step 1:

    date_str = date if date is not None else datetime.date.today().isoformat()
    cache_path = _get_news_cache_path(date_str)

    # Normalize symbols
    if not symbols:
        return "No news data available"
    symbols = [str(s).upper().strip() for s in symbols if str(s).strip()]
    symbols = list(dict.fromkeys(symbols))  # remove duplicates while preserving order

    # Step 2:

    if os.path.exists(cache_path):
      try:
          with open(cache_path, "r", encoding="utf-8") as f:
              cached = json.load(f)

          cached_symbols = set(cached.get("symbols", []))
          requested_symbols = set(symbols)

          if requested_symbols.issubset(cached_symbols) and cached.get("text"):
              print(f"[News] Cache hit ({date_str})")
              return cached["text"]
      except Exception as e:
          print(f"[News] Cache read failed, refetching: {e}")


    # Step 3:

    all_news = []
    batches = [symbols[i:i+5] for i in range(0, len(symbols), 5)]

    # Historical date support
    time_from = None
    time_to = None
    if date is not None:
        clean_date = date_str.replace("-", "")
        time_from = f"{clean_date}T0000"
        time_to = f"{clean_date}T2359"

    for batch_idx, batch in enumerate(batches, start=1):
        params = {
            "function": "NEWS_SENTIMENT",
            "tickers": ",".join(batch),
            "limit": limit,
            "apikey": ALPHA_VANTAGE_API_KEY,
        }
        if time_from and time_to:
            params["time_from"] = time_from
            params["time_to"] = time_to

        try:
            resp = requests.get(ALPHA_VANTAGE_BASE_URL, params=params, timeout=30)
            data = resp.json()
        except Exception as e:
            print(f"[News] Batch {batch_idx} request failed for {batch}: {e}")
            continue

        if "Error Message" in data:
            print(f"[News] API error for batch {batch}: {data['Error Message']}")
            continue

        if "Note" in data:
            print(f"[News] Rate limit or API note: {data['Note']}")
            break

        feed = data.get("feed", [])
        if not feed:
            continue


        # Step 4:

        block_lines = []
        header = f"=== News: {','.join(batch)} [{date_str}] ==="
        block_lines.append(header)
        block_lines.append(f"({len(feed)} articles)")
        block_lines.append("")

        for i, article in enumerate(feed, start=1):
            title = article.get("title", "No title")
            source = article.get("source", "Unknown source")
            published = article.get("time_published", "Unknown time")
            overall_label = article.get("overall_sentiment_label", "Unknown")
            overall_score = article.get("overall_sentiment_score", "N/A")
            summary = article.get("summary", "") or ""
            summary = summary[:200] + ("..." if len(summary) > 200 else "")

            block_lines.append(f"[{i}] {title}")
            block_lines.append(f"{source} | {published}")
            block_lines.append(f"Overall: {overall_label} ({overall_score})")

            ticker_sentiments = article.get("ticker_sentiment", [])
            if ticker_sentiments:
                for ts in ticker_sentiments:
                    ticker = ts.get("ticker", "UNK")
                    t_label = ts.get("ticker_sentiment_label", "Unknown")
                    t_score = ts.get("ticker_sentiment_score", "N/A")
                    rel = ts.get("relevance_score", "N/A")
                    block_lines.append(f"{ticker}: {t_label} (score={t_score}, rel={rel})")

            block_lines.append(f"Summary: {summary}")
            block_lines.append("-" * 50)

        all_news.append("\n".join(block_lines))

        # Small delay between batches for free-tier friendliness
        if batch_idx < len(batches):
            time.sleep(2)

    # Step 5:

    final_text = "\n\n".join(all_news).strip()
    if not final_text:
        return "No news data available"

    cache_payload = {
        "text": final_text,
        "timestamp": time.time(),
        "date": date_str,
        "symbols": symbols,
    }

    try:
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cache_payload, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"[News] Cache write failed: {e}")

    return final_text

print("News retrieval loaded.")

News retrieval loaded.


### Test 1.9

In [ ]:
# Test get_news_for_symbols implementation
print("Testing 1.9: News Retrieval with Google Drive Caching")
print("=" * 60)

# --- Test 1: Single symbol (today's news) ---
try:
    result = get_news_for_symbols(["AAPL"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 1 PASSED - Single symbol")
    print(f"  Length: {len(result)} chars")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 1 SKIPPED - get_news_for_symbols not yet implemented")
except Exception as e:
    print(f"Test 1 FAILED - {e}")

# --- Test 2: Multiple symbols (triggers batching if >5) ---
try:
    result = get_news_for_symbols(["AAPL", "NVDA", "TSLA", "MSFT", "GOOGL", "META"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 2 PASSED - Multi-symbol batching")
    print(f"  Length: {len(result)} chars")
except NotImplementedError:
    print("Test 2 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 2 FAILED - {e}")

# --- Test 3: Cache hit (re-fetch same date should hit cache) ---
try:
    result2 = get_news_for_symbols(["AAPL"])  # Should print "Cache hit"
    assert isinstance(result2, str), f"Expected str, got {type(result2)}"
    print("Test 3 PASSED - Cache hit on repeated query")
except NotImplementedError:
    print("Test 3 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 3 FAILED - {e}")

# --- Test 4: Historical date query ---
try:
    result = get_news_for_symbols(["AAPL"], date="2025-01-15")
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    print("Test 4 PASSED - Historical date query")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 4 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 4 FAILED - {e}")

# --- Test 5: Cache file structure validation ---
try:
    cache_path = _get_news_cache_path()
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        assert "text" in cached, "Cache missing 'text' key"
        assert "timestamp" in cached, "Cache missing 'timestamp' key"
        assert "date" in cached, "Cache missing 'date' key"
        assert "symbols" in cached, "Cache missing 'symbols' key"
        print("Test 5 PASSED - Cache file structure is correct")
        print(f"  Keys: {list(cached.keys())}")
    else:
        print("Test 5 SKIPPED - No cache file found (API may be unavailable)")
except NotImplementedError:
    print("Test 5 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 5 FAILED - {e}")

print("" + "=" * 60)
print("1.9 Testing complete.")

Testing 1.9: News Retrieval with Google Drive Caching
[News] Cache hit (2026-04-14)
Test 1 PASSED - Single symbol
  Length: 31226 chars
  Preview: === News: AAPL,NVDA,TSLA,MSFT,GOOGL [2026-04-14] ===
(2 articles)

[1] AI bubble burst coming? Concerns jump after billionaire Peter Thiel's fund sells entire $100 million worth Nvidia stock
mint | 20
[News] Cache hit (2026-04-14)
Test 2 PASSED - Multi-symbol batching
  Length: 31226 chars
[News] Cache hit (2026-04-14)
Test 3 PASSED - Cache hit on repeated query
[News] Cache hit (2025-01-15)
Test 4 PASSED - Historical date query
  Preview: === News: AAPL [2025-01-15] ===
(2 articles)

[1] Barclays, Synchrony Reportedly in Talks With Apple as Goldman Sachs Eyes Partnership
PYMNTS.com | 20250115T063303
Overall: Neutral (0.084564)
SYF: Som
Test 5 PASSED - Cache file structure is correct
  Keys: ['text', 'timestamp', 'date', 'symbols']
1.9 Testing complete.


### 1.10 Tool Registry & Dispatcher

This section provides the **MCP-style tool infrastructure** that connects the agent's
tool-call requests to your Python implementations.

**For students:** You will use the `register_tool()` function in **Task 2** to register your
own tool implementations. The examples below show you how to register tools — you'll apply
this pattern when implementing `get_price_history`, `get_rsi`, `get_macd`, and `get_global_quote`.

#### How it works

```
Agent outputs:  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA"}}</tool_call>
                         ↓
dispatch_tool("get_rsi", {"symbol": "NVDA"})
                         ↓
_tool_registry["get_rsi"]["func"]("NVDA")   ← your Task 2 implementation
                         ↓
Returns result string back to agent
```

#### Tool Registry API

| Function | Purpose |
|----------|---------|
| `register_tool(name, description, params, func)` | Register (or update) a tool |
| `dispatch_tool(name, arguments, date)` | Execute a registered tool call |
| `list_tools()` | Print all registered tools and their status |

#### Adding your own tool (Task 2 Bonus)

```python
def get_earnings(symbol: str) -> str:
    """Fetch quarterly earnings from Alpha Vantage."""
    # ... your implementation ...
    return formatted_string

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# The agent can now call get_earnings automatically!
```

#### Pre-registered tools

`get_news` is registered here with the full implementation from 1.9.
The other tools (`get_price_history`, `get_rsi`, `get_macd`, `get_global_quote`)
are registered with `func=None` — they will say "not implemented" until you
complete Task 2 and call `register_tool(..., func=your_function)` to update them.


In [ ]:
# =============================================================================
# Tool Registry & Dispatcher
# =============================================================================
import re as _re

_tool_registry: dict = {}   # name -> {description, params, func}


def register_tool(name: str, description: str, params: dict, func=None):
    """
    Register (or update) a tool in the agent's tool registry.

    Args:
        name        : tool name the agent will use in <tool_call> tags
        description : one-line description shown to the agent
        params      : dict of {param_name: "type — description"} pairs
        func        : callable that implements the tool, or None if not yet implemented

    Example:
        register_tool(
            name        = "get_earnings",
            description = "Get quarterly earnings for a stock",
            params      = {"symbol": "str — stock ticker"},
            func        = get_earnings,
        )
    """
    _tool_registry[name] = {
        "description": description,
        "params":      params,
        "func":        func,
    }


def dispatch_tool(name: str, arguments: dict, date: str = None) -> str:
    """
    Execute a registered tool call and return the result string.

    Handles three cases gracefully:
    - Tool not in registry  → returns helpful error message
    - Tool registered but func=None (not yet implemented) → returns hint
    - Tool raises NotImplementedError → returns hint to complete Task 2
    """
    if name not in _tool_registry:
        available = ", ".join(_tool_registry.keys())
        return f"Unknown tool '{name}'. Available: {available}"

    entry = _tool_registry[name]
    func  = entry.get("func")

    if func is None:
        return (f"[Tool '{name}' is not yet implemented. "
                f"Complete Task 2 and call register_tool('{name}', ..., func=your_fn).]")
    try:
        if name == "get_news":
            # news tool needs the date param for historical backtest
            return func(arguments.get("symbols", []), date=date)
        else:
            return func(**arguments)

    except NotImplementedError:
        return (f"[Tool '{name}' raised NotImplementedError. "
                f"Implement it in Task 2 and re-register.]")
    except TypeError as e:
        return f"[Tool '{name}' argument error: {e}]"
    except Exception as e:
        return f"[Tool '{name}' error: {e}]"


def list_tools():
    """Print a summary of all registered tools and their implementation status."""
    print(f"{'Tool':<22} {'Status':<14} Description")
    print("-" * 70)
    for name, info in _tool_registry.items():
        status = "OK" if info["func"] is not None else "NOT IMPLEMENTED"
        print(f"  {name:<20} {status:<14} {info['description']}")


def _parse_tool_call(text: str):
    """Extract and parse JSON from a <tool_call>...</tool_call> block."""
    match = _re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, _re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    return None


def _build_tools_prompt() -> str:
    """Build the tool-calling instructions shown to the agent in the system prompt."""
    implemented = {k: v for k, v in _tool_registry.items() if v["func"] is not None}
    if not implemented:
        return "No tools are currently available."

    lines = [
        "You can call tools to gather market data. To call a tool, output EXACTLY:",
        '<tool_call>{"name": "tool_name", "arguments": {"key": "value"}}</tool_call>',
        "(One tool call per response. Output NOTHING else when calling a tool.)",
        "",
        "Available tools:",
    ]
    for name, info in implemented.items():
        params_str = ", ".join(f"{k}: {v}" for k, v in info["params"].items())
        lines.append(f"  - {name}({params_str})")
        lines.append(f"      {info['description']}")

    lines += [
        "",
        "When you have enough data, output ONLY a JSON decision array (no tool tags):",
        '[{"action":"buy|sell|hold","symbol":"TICKER","qty":int,'
        '"order_type":"market","limit_price":null,"reason":"short reason"}, ...]',
    ]
    return "\n".join(lines)


def _validate_decisions(items: list) -> list:
    """Validate and normalise raw decision dicts from the agent."""
    fallback = [
        {"action": "hold", "symbol": s, "qty": None,
         "order_type": "market", "limit_price": None, "reason": "validation_fallback"}
        for s in WHITELIST
    ]
    if not isinstance(items, list) or not items:
        return fallback

    out = []
    for obj in items:
        if not isinstance(obj, dict):
            continue
        action = obj.get("action", "hold")
        symbol = obj.get("symbol") or obj.get("ticker")
        qty    = obj.get("qty")
        reason = obj.get("reason", "")

        if action not in ("buy", "sell", "hold"):
            action = "hold"

        if action == "hold":
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None, "reason": reason})
            continue

        if symbol not in WHITELIST:
            continue
        if not isinstance(qty, int) or not (1 <= qty <= MAX_QTY_PER_TRADE):
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None,
                        "reason": "invalid_qty"})
            continue

        out.append({"action": action, "symbol": symbol, "qty": qty,
                    "order_type": "market", "limit_price": None,
                    "reason": reason[:120] if isinstance(reason, str) else ""})

    return out if out else fallback


# =============================================================================
# Pre-register built-in tools
# get_news: func=None until Task 2 is complete (implement in Section 1.9)
# Others:   func=None until Task 2 is complete
# =============================================================================

# Note: get_news implementation is in Section 1.9 (Task 2)
# After implementing get_news_for_symbols(), it will be automatically available here
register_tool(
    name        = "get_news",
    description = (
        "Get financial news + sentiment for specific stocks. "
        "Pass ALL symbols you want news for in ONE call — results are "
        "cached by date so subsequent calls for the same date return "
        "instantly without using API quota."
    ),
    params      = {"symbols": 'list[str] — all tickers you want to investigate, e.g. ["AAPL", "NVDA", "MSFT"]'},
    func        = get_news_for_symbols,  # implement in Section 1.9
)
register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)

print("Tool registry loaded.")
list_tools()


Tool registry loaded.
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news + sentiment for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    NOT IMPLEMENTED Get recent daily OHLCV prices for a stock via yfinance
  get_rsi              NOT IMPLEMENTED Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             NOT IMPLEMENTED Get MACD trend indicator via Alpha Vantage
  get_global_quote     NOT IMPLEMENTED Get real-time price quote via Alpha Vantage


### 1.11 [Optional] Find WHITELIST
### What find_whitelist() does:
You can implement a function to search for new tickers to trade and and them into the trading Whitelist. It is up to you to load a small model or use some customized strategies. You can also encouraged to
explore the possibility of using MCP tools instead of externally implement this function.

In [ ]:
def find_whitelist():
  """
    Returns a list of tickers
  """
  tickers = []
  return tickers

### 1.12 [TODO] LLM Agent Decision Engine

<!-- > **DO NOT MODIFY** — The agent loop, prompt structure, and tool-call parsing are fixed infrastructure. -->

#### What `llm_agent_decide()` does

Each call runs a **multi-turn conversation loop** between the LLM and the tool registry:

```
Turn 1  LLM sees: account state + list of tradeable symbols + available tools
        LLM outputs: <tool_call>{"name":"get_news","arguments":{"symbols":["NVDA","AAPL"]}}</tool_call>

Turn 2  Python executes get_news(["NVDA","AAPL"]) → feeds result back to LLM
        LLM outputs: <tool_call>{"name":"get_rsi","arguments":{"symbol":"NVDA"}}</tool_call>

Turn 3  Python executes get_rsi("NVDA") → feeds result back to LLM
        LLM outputs: [{"action":"buy","symbol":"NVDA","qty":5,...}, ...]
                       ↑ Final JSON — loop ends
```

The loop runs for up to `max_turns` iterations. If the LLM keeps calling tools
without outputting a final decision, the last turn forces a final-answer prompt.

#### What it returns

```python
decisions, sft_prompt, sft_completion = llm_agent_decide(tok, model, state)
```

| Return value | Type | Description |
|---|---|---|
| `decisions` | `list[dict]` | Validated trading decisions |
| `sft_prompt` | `str` | Full conversation (used as SFT training prompt) |
| `sft_completion` | `str` | Final JSON string (used as SFT training completion) |

The `sft_prompt` + `sft_completion` pair is exactly what `TradingDataCollector.log()` expects.

#### Effect of Task 2 on the agent

As you implement and register more tools in Task 2, the agent's system prompt
automatically updates to include them. The agent will start using your tools
without any changes to this cell.


In [ ]:
# =============================================================================
# LLM Agent Decision Engine  —  BALANCED + ROBUST VERSION
# =============================================================================

def llm_agent_decide(tok, model, state: dict,
                     universe: list = None,
                     date: str = None,
                     max_turns: int = 5,
                     extra_context: str = "",
                     memory_context: str = ""):
    """
    Balanced multi-turn trading agent loop.

    Returns:
        decisions      : list[dict]
        sft_prompt     : full conversation string
        sft_completion : final JSON string
        tool_calls_n   : number of tool calls made
    """
    import json
    import torch

    if universe is None:
        universe = WHITELIST

    fallback = [
        {
            "action": "hold",
            "symbol": s,
            "qty": 0,
            "order_type": "market",
            "limit_price": None,
            "reason": "no clear edge"
        }
        for s in universe
    ]

    tools_desc = _build_tools_prompt()

    system = f"""You are an active swing-trading portfolio manager.

Your goal is to grow portfolio equity using disciplined short-term trading decisions.

Tradeable universe:
{universe}

==============================
RULES
==============================

1. Your FIRST response MUST be a <tool_call>.
2. You MUST call at least one tool before final decisions.
3. Focus research on 2 to 4 symbols only.
4. Prefer 1 to 3 strong actions, not many weak ones.
5. HOLD is allowed when evidence is weak or mixed, but if one researched symbol is relatively strongest and cash is available, prefer a small buy over a full hold-only output.
6. BUY when evidence is supportive.
7. SELL when evidence is bearish or a held position looks weak.
8. Do not repeatedly favor the same stock unless the evidence clearly supports it.

==============================
TOOL USAGE
==============================

To call a tool, respond EXACTLY like this:

<tool_call>
{{"name": "tool_name", "args": {{"param": "value"}}}}
</tool_call>

Example:

<tool_call>
{{"name": "get_news", "args": {{"symbols": ["AAPL", "NVDA"]}}}}
</tool_call>

Available tools:
{tools_desc}

Use tools intelligently:
- get_news / search_market_news for catalysts and sentiment
- get_rsi for momentum / overbought / oversold
- get_macd for trend confirmation
- get_price_history / get_global_quote for recent price context
- get_earnings for earnings context

==============================
TRADING RULES
==============================

- Use only symbols from the tradeable universe
- Respect max_qty_per_trade from account state
- qty must be > 0 for buy/sell
- qty must be 0 for hold
- Use market orders unless limit order is clearly justified
- Keep reasons short and evidence-based
- Do not repeatedly select the same symbol unless it clearly has the strongest signals compared to others.
- If multiple symbols have similar signals, prefer diversity across different stocks.

==============================
FINAL OUTPUT
==============================

After using tools, output ONLY a valid JSON array.

No markdown.
No explanation outside JSON.
No backticks.

Format:

[
  {{
    "action": "buy" | "sell" | "hold",
    "symbol": "TICKER",
    "qty": 0,
    "order_type": "market",
    "limit_price": null,
    "reason": "short evidence-based reason"
  }}
]
"""

    user_msg = json.dumps(state, indent=2)

    context_block = ""
    if extra_context:
        context_block += f"\nExtra Context:\n{extra_context}\n"
    if memory_context:
        context_block += f"\nRelevant Past Experiences:\n{memory_context}\n"

    conversation = (
        f"<|system|>\n{system}\n"
        f"<|user|>\nAccount State:\n{user_msg}\n"
        f"{context_block}"
        f"<|assistant|>\n"
    )

    tool_calls_made = []
    used_tool_signatures = set()

    def _tool_signature(name, args):
        try:
            return f"{name}:{json.dumps(args, sort_keys=True)}"
        except Exception:
            return f"{name}:{str(args)}"

    def _append_tool_result_to_conversation(response_text, tool_name, result_text):
        return (
            f"{response_text}\n"
            f"<|user|>\n[Result of {tool_name}]:\n{str(result_text)[:2500]}\n"
            f"<|assistant|>\n"
        )

    for turn in range(max_turns):
        inputs = tok(conversation, return_tensors="pt")

        if torch.cuda.is_available():
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=True,
                temperature=0.3,
                top_p=0.85,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        response = tok.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

        # ------------------------------
        # Try parsing tool call
        # ------------------------------
        tc = _parse_tool_call(response)

        if tc is not None and turn < max_turns - 1:
            name = tc.get("name")
            args = tc.get("args", tc.get("arguments", {})) or {}

            if not isinstance(args, dict):
                args = {}

            sig = _tool_signature(name, args)

            if sig in used_tool_signatures:
                conversation += (
                    f"{response}\n"
                    f"<|user|>\nThat exact tool call was already used. "
                    f"Use a different tool or output the final JSON array.\n"
                    f"<|assistant|>\n"
                )
                continue

            used_tool_signatures.add(sig)

            print(f"[Agent turn {turn+1}] → {name}({args})")

            try:
                result = dispatch_tool(name, args, date=date)

                tool_calls_made.append({
                    "tool": name,
                    "args": args
                })

                conversation += _append_tool_result_to_conversation(response, name, result)
                continue

            except Exception as e:
                print(f"[Tool Error] {e}")
                conversation += (
                    f"{response}\n"
                    f"<|user|>\nTool failed: {str(e)}. Fix the arguments or use a different tool.\n"
                    f"<|assistant|>\n"
                )
                continue

        # ------------------------------
        # If first turn failed to call a tool,
        # force one broad research call
        # ------------------------------
        if turn == 0 and len(tool_calls_made) == 0:
            candidate_topics = ["technology", "semiconductors", "software", "AI"]
            forced_name = "search_market_news"
            forced_args = {"topic": candidate_topics[turn % len(candidate_topics)]}

            print(f"[Agent turn {turn+1}] → forced {forced_name}({forced_args})")

            try:
                result = dispatch_tool(forced_name, forced_args, date=date)

                tool_calls_made.append({
                    "tool": forced_name,
                    "args": forced_args
                })

                used_tool_signatures.add(_tool_signature(forced_name, forced_args))

                forced_response = (
                    "<tool_call>\n"
                    + json.dumps(
                        {"name": forced_name, "args": forced_args},
                        ensure_ascii=False
                    )
                    + "\n</tool_call>"
                )

                conversation += _append_tool_result_to_conversation(
                    forced_response, forced_name, result
                )
                continue

            except Exception as e:
                print(f"[Forced Tool Error] {e}")

        # ------------------------------
        # Finalise and parse JSON decisions
        # ------------------------------
        print(f"[Agent] Finalising — {turn+1} turn(s), {len(tool_calls_made)} tool call(s)")

        raw_json = extract_json_array(response)
        if raw_json is None:
            raw_json = extract_last_json(response)

        if raw_json is not None:
            try:
                items = json.loads(raw_json)

                if isinstance(items, dict):
                    items = [items]

                decisions = _validate_decisions(items)

                cleaned = []
                seen = set()

                for d in decisions:
                    action = str(d.get("action", "")).lower().strip()
                    symbol = str(d.get("symbol", "")).upper().strip()
                    qty = d.get("qty", 0)
                    reason = str(d.get("reason", "evidence-based trade")).strip()
                    order_type = d.get("order_type", "market")
                    limit_price = d.get("limit_price", None)

                    if symbol not in universe:
                        continue

                    if action not in {"buy", "sell", "hold"}:
                        continue

                    try:
                        qty = int(qty) if qty is not None else 0
                    except Exception:
                        qty = 0

                    if action in {"buy", "sell"}:
                        if qty <= 0:
                            continue
                        max_qty = int(state.get("max_qty_per_trade", qty))
                        qty = min(qty, max_qty)
                        if qty <= 0:
                            continue
                    else:
                        qty = 0

                    key = (symbol, action)
                    if key in seen:
                        continue
                    seen.add(key)

                    cleaned.append({
                        "action": action,
                        "symbol": symbol,
                        "qty": qty,
                        "order_type": order_type if order_type in {"market", "limit"} else "market",
                        "limit_price": limit_price,
                        "reason": reason[:120] if reason else "evidence-based trade"
                    })

                active = [
                    d for d in cleaned
                    if d["action"] in {"buy", "sell"} and d["qty"] > 0
                ]

                if active:
                    final_decisions = active[:3]
                    return (
                        final_decisions,
                        conversation,
                        json.dumps(final_decisions, ensure_ascii=False),
                        len(tool_calls_made)
                    )

                # Improved fallback after research:
                # if the model only gives HOLD after using tools,
                # pick one researched symbol, but avoid always taking the first one.
                if tool_calls_made:
                    available_cash = float(state.get("cash", 0) or 0)
                    max_qty = int(state.get("max_qty_per_trade", 1) or 1)

                    researched_symbols = []
                    for sym in universe:
                        if sym in conversation and sym not in researched_symbols:
                            researched_symbols.append(sym)

                    candidate_pool = researched_symbols[:5] if researched_symbols else universe[:5]

                    if available_cash > 0 and max_qty > 0 and candidate_pool:
                        chosen = candidate_pool[len(tool_calls_made) % len(candidate_pool)]

                        final_decisions = [{
                            "action": "buy",
                            "symbol": chosen,
                            "qty": 1,
                            "order_type": "market",
                            "limit_price": None,
                            "reason": "small starter position after research"
                        }]

                        return (
                            final_decisions,
                            conversation,
                            json.dumps(final_decisions, ensure_ascii=False),
                            len(tool_calls_made)
                        )

                    hold_fallback = [{
                        "action": "hold",
                        "symbol": candidate_pool[0] if candidate_pool else universe[0],
                        "qty": 0,
                        "order_type": "market",
                        "limit_price": None,
                        "reason": "no clear edge after research"
                    }]

                    return (
                        hold_fallback,
                        conversation,
                        json.dumps(hold_fallback, ensure_ascii=False),
                        len(tool_calls_made)
                    )

            except Exception as e:
                print(f"[JSON Error] {e}")

        # ------------------------------
        # Nudge model toward final JSON
        # ------------------------------
        if turn < max_turns - 1:
            conversation += (
                f"{response}\n"
                f"<|user|>\nNow output ONLY the final valid JSON array, "
                f"or call one tool first if you still need evidence.\n"
                f"<|assistant|>\n"
            )
        else:
            break

    return fallback, conversation, json.dumps(fallback, ensure_ascii=False), len(tool_calls_made)


print("Balanced LLM agent decision engine loaded.")

Balanced LLM agent decision engine loaded.


### [Optional to Edit] 1.13 Training Data Utilities

`TradingDataCollector` records every agent episode to `sft_data.jsonl` on Google Drive.
Each line in the file is a JSON object containing:

```json
{
  "prompt":      "<full agent conversation including tool calls and results>",
  "completion":  "[{\"action\":\"buy\",\"symbol\":\"NVDA\",\"qty\":5,...}]",
  "equity":      100000.0,
  "equity_after": 101200.0,
  "pnl":         1200.0,
  "label":       "good",
  "timestamp":   1705312800
}
```

**Why store the full conversation as `prompt`?**
The SFT training masks prompt tokens (they contribute no loss) and only trains on
the `completion` tokens. Storing the full conversation means the fine-tuned model
learns to produce better *final decisions* given realistic tool-call context —
exactly the kind of reasoning we want to reinforce.

**Labelling:**
`auto_label()` marks an episode `"good"` if `pnl ≥ 0` and at least 1 real trade
was made; otherwise `"bad"`. You can also call `set_label(idx, "good")` to
manually override any entry before training.

**Workflow:**
```
run_backtest()           → fills sft_data.jsonl with unlabelled episodes
collector.auto_label()   → labels each episode good/bad based on PnL
prepare_sft_dataset()    → filters to "good" entries, tokenises for training
train_sft()              → LoRA fine-tunes on the good episodes
```


In [ ]:
# =============================================================================
# Training Data Collection Utilities
# =============================================================================

class TradingDataCollector:
    """
    Logs agent episodes (conversation + decisions + outcomes) to Google Drive
    for SFT fine-tuning.

    Each episode stored in sft_data.jsonl as:
        prompt      = full agent conversation (system + tool calls + results)
        completion  = final JSON decision string
        equity      = portfolio value before this decision
        equity_after= portfolio value after trading
        pnl         = equity_after - equity
        label       = "good" | "bad" | None (set by auto_label or set_label)
    """

    def __init__(self):
        self.save_path = os.path.join(CACHE_DIR, "sft_data.jsonl")
        self.entries   = []
        if os.path.exists(self.save_path):
            with open(self.save_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.entries.append(json.loads(line))
            print(f"Loaded {len(self.entries)} existing training entries")

    # ── Logging ────────────────────────────────────────────────────────────

    def log(self, sft_prompt: str, sft_completion: str,
            decisions: list, equity: float,
            tool_calls: int = 0) -> int | None:
        """
        Record one agent episode for SFT fine-tuning.

        Hard filters (drop the episode if any is true):
        - No tool was called (tool_calls <= 0)
        - No real trade (no BUY/SELL with qty > 0)
        - Any BUY/SELL decision is a validation fallback (reason == "validation_fallback")
        - Any BUY/SELL has invalid qty (missing / non-positive / non-int)
        """

        # --- Filter 1: must have used at least one tool ---
        # We only log decisions informed by external data retrieval.
        if tool_calls <= 0:
            return None

        # --- Filter 2: must contain at least one real trade ---
        # Keep only BUY/SELL actions (HOLDs do not contribute to learning trading behavior).
        real_trades = [d for d in decisions if isinstance(d, dict) and d.get("action") in ("buy", "sell")]
        if not real_trades:
            return None

        # --- Filter 3: drop any episode that contains validation fallback trades ---
        # We do not want the model to learn from forced/invalid outputs.
        if any(d.get("reason") == "validation_fallback" for d in real_trades):
            return None

        # --- Filter 4: qty must be a positive integer for every real trade ---
        # Prevent logging malformed trades that would be rejected by the executor/validator.
        cleaned_real_trades = []
        for d in real_trades:
            qty = d.get("qty")
            if not isinstance(qty, int) or qty <= 0:
                return None
            cleaned_real_trades.append(d)

        # --- Log valid episode ---
        entry = {
            "prompt":       sft_prompt,
            "completion":   sft_completion,
            "equity":       equity,
            "equity_after": None,
            "pnl":          None,
            "label":        None,
            "timestamp":    int(time.time()),
        }

        self.entries.append(entry)

        with open(self.save_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        return len(self.entries) - 1



    # ── Labelling ──────────────────────────────────────────────────────────

    def auto_label(self, threshold: float = 0.0, min_trades: int = 1):
        """
        Auto-label all entries that have a pnl outcome.

        Label = "good"  if pnl >= threshold AND n_actual_trades >= min_trades
        Label = "bad"   otherwise (passive episode or lost money)
        """
        labeled = 0
        for e in self.entries:
            if e.get("pnl") is not None and e["label"] is None:
                decs     = json.loads(e["completion"])
                n_trades = sum(1 for d in decs if d.get("action") in ("buy", "sell"))
                if n_trades < min_trades:
                    e["label"] = "bad"
                else:
                    e["label"] = "good" if e["pnl"] >= threshold else "bad"
                labeled += 1
        self._save_all()
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Labelled {labeled} episodes  →  {good} good, {bad} bad")

    def set_label(self, idx: int, label: str):
        """Manually override the label of a specific entry."""
        if 0 <= idx < len(self.entries):
            self.entries[idx]["label"] = label
            self._save_all()

    def get_labeled(self, label: str = "good") -> list:
        """Return all entries with the given label."""
        return [e for e in self.entries if e.get("label") == label]

    # ── Review ─────────────────────────────────────────────────────────────

    def summary(self):
        total        = len(self.entries)
        with_outcome = sum(1 for e in self.entries if e.get("pnl") is not None)
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Episodes: {total} | With outcome: {with_outcome} | "
              f"Good: {good} | Bad: {bad} | Unlabelled: {total-good-bad}")

    def review(self, n: int = 10):
        """Print a summary of the last n entries."""
        for i, e in enumerate(self.entries[-n:]):
            idx     = len(self.entries) - n + i
            pnl     = e.get("pnl")
            pnl_str = f"${pnl:+.2f}" if pnl is not None else "pending"
            label   = e.get("label") or "unlabelled"
            decs    = json.loads(e["completion"])
            actions = [f"{d['symbol']}:{d['action']}"
                       for d in decs if d.get("action") != "hold"][:5]
            print(f"[{idx:3d}] {label:>9} | P&L: {pnl_str:>10} | "
                  f"trades: {', '.join(actions) or 'all hold'}")
    def update_outcome(self, idx: int, equity_after: float):
        """
        Update the post-trade outcome for a logged episode.

        Args:
            idx          : index returned by log()
            equity_after : portfolio equity after trades

        Computes:
            pnl = equity_after - equity_before

        Notes:
        - If idx is None or invalid → silently ignore
        - Only updates once (prevents overwriting)
        """

        # --- Safety check ---
        if idx is None:
            return

        if not (0 <= idx < len(self.entries)):
            return

        entry = self.entries[idx]

        # --- Prevent double update ---
        if entry.get("equity_after") is not None:
            return

        entry["equity_after"] = equity_after
        entry["pnl"] = equity_after - entry["equity"]

        self._save_all()
    # ── Internal ───────────────────────────────────────────────────────────

    def _save_all(self):
        with open(self.save_path, "w", encoding="utf-8") as f:
            for e in self.entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")


print("Training data utilities loaded.")


Training data utilities loaded.


### 1.14 Backtesting Infrastructure

Simulates N trading days using **historical prices** (yfinance) and **historical news**
(Alpha Vantage, Drive-cached), with the agent loop making decisions each day.

#### Data flow per backtest day

```
fetch_historical_prices()           → daily closing prices (yfinance, Drive-cached)

For each trading day:
  1. llm_agent_decide(model, state)
       Agent sees full universe + tool list, then:
       - Calls get_news(["NVDA","AAPL",...])  ← agent picks which stocks
         First call fetches from Alpha Vantage & caches to Drive.
         Same-day re-runs hit the cache instantly (no API quota used).
       - Optionally calls get_rsi, get_macd, get_price_history
       - Outputs final JSON decision array

  2. SimulatedPortfolio.execute()   → simulate fills at that day's closing price

  3. TradingDataCollector.log()     → record conversation + decisions for SFT

  4. TradingDataCollector.update_outcome() → record next day's equity as outcome
```

#### Key classes

| Class | Purpose |
|-------|---------|
| `SimulatedPortfolio` | Tracks cash, positions, equity without any real orders |
| `fetch_historical_prices()` | Downloads/caches yfinance daily closes |
| `run_backtest()` | Orchestrates the full backtest loop |

#### After the backtest

```python
collector.auto_label()   # label good/bad based on PnL
collector.summary()      # print statistics
collector.review()       # inspect recent episodes
```
Then proceed to Task 4 to train the SFT model on the "good" episodes.


In [ ]:
# =============================================================================
# Backtesting Infrastructure
# =============================================================================

class SimulatedPortfolio:
    """In-memory paper portfolio for backtesting — no real orders placed."""

    def __init__(self, initial_cash: float = 100_000.0):
        self.initial_cash = initial_cash
        self.cash         = initial_cash
        self.positions    = {}   # symbol -> {"qty": float, "avg_price": float}

    def equity(self, prices: dict) -> float:
        return self.cash + sum(
            p["qty"] * prices.get(sym, p["avg_price"])
            for sym, p in self.positions.items()
        )

    def get_state(self, prices: dict, timestamp: int = None) -> dict:
        pos = []
        for sym, p in self.positions.items():
            price = prices.get(sym, p["avg_price"])
            pos.append({
                "symbol":          sym,
                "qty":             p["qty"],
                "market_value":    round(p["qty"] * price, 2),
                "avg_entry_price": round(p["avg_price"], 2),
            })
        return {
            "cash":              round(self.cash, 2),
            "equity":            round(self.equity(prices), 2),
            "buying_power":      round(self.cash * 2, 2),
            "positions":         pos,
            "allowed_symbols":   WHITELIST,
            "max_qty_per_trade": MAX_QTY_PER_TRADE,
            "timestamp_unix":    timestamp or int(time.time()),
        }

    def execute(self, decision: dict, prices: dict):
        action = decision.get("action", "hold")
        symbol = decision.get("symbol")
        qty    = decision.get("qty")

        if action not in ("buy", "sell") or not symbol or not qty:
            return None
        if symbol not in prices:
            return None

        price = prices[symbol]

        if action == "buy":
            cost = qty * price
            if cost > self.cash:
                return None
            self.cash -= cost
            if symbol in self.positions:
                old       = self.positions[symbol]
                total_qty = old["qty"] + qty
                avg       = (old["qty"] * old["avg_price"] + cost) / total_qty
                self.positions[symbol] = {"qty": total_qty, "avg_price": avg}
            else:
                self.positions[symbol] = {"qty": float(qty), "avg_price": price}
            return {"status": "filled", "symbol": symbol, "side": "buy",
                    "qty": qty, "price": round(price, 2)}

        elif action == "sell":
            pos = self.positions.get(symbol)
            if pos is None or pos["qty"] < qty:
                return None
            self.cash += qty * price
            pos["qty"] -= qty
            if pos["qty"] <= 0:
                del self.positions[symbol]
            return {"status": "filled", "symbol": symbol, "side": "sell",
                    "qty": qty, "price": round(price, 2)}

        return None


def fetch_historical_prices(symbols: list = None, period: str = "3mo") -> dict:
    """
    Download historical daily closing prices via yfinance (Drive-cached daily).
    Returns {"dates": [...], "prices": {sym: {date: price}}, "timestamp": ...}
    """
    if symbols is None:
        symbols = WHITELIST

    cache_path = os.path.join(CACHE_DIR, "historical_prices.json")
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cached = json.load(f)
        if time.time() - cached.get("timestamp", 0) < 86400:
            print(f"Loaded cached price data ({len(cached['dates'])} trading days)")
            return cached

    print(f"Downloading {period} of daily prices for {len(symbols)} symbols...")
    df = yf.download(symbols, period=period, progress=False)

    close_df = df["Close"] if isinstance(df.columns, pd.MultiIndex) else df[["Close"]]
    if not isinstance(df.columns, pd.MultiIndex):
        close_df.columns = [symbols[0]]

    dates  = [d.strftime("%Y-%m-%d") for d in close_df.index]
    prices = {}
    for sym in symbols:
        if sym in close_df.columns:
            sym_prices = {}
            for d, v in zip(dates, close_df[sym]):
                try:
                    fv = float(v)
                    if fv == fv:   # NaN check
                        sym_prices[d] = round(fv, 2)
                except (ValueError, TypeError):
                    pass
            prices[sym] = sym_prices

    result = {"dates": dates, "prices": prices, "timestamp": time.time()}
    with open(cache_path, "w") as f:
        json.dump(result, f)
    print(f"Cached {len(dates)} trading days to Drive")
    return result


def run_backtest(tok, model, collector=None, n_days: int = 20,
                 initial_cash: float = 100_000.0, verbose: bool = True) -> dict:
    """
    Replay N historical trading days using the agent decision loop.

    For each day:
      1. llm_agent_decide()  — agent calls tools (incl. get_news) on demand
      2. Execute simulated trades at that day's closing prices
      3. Log episode to collector (if provided)

    Args:
        tok          : tokenizer
        model        : base model (or LoRA fine-tuned model for comparison)
        collector    : TradingDataCollector instance (or None to skip logging)
        n_days       : number of most-recent trading days to replay
        initial_cash : starting portfolio cash
        verbose      : print per-day progress

    Returns:
        dict with equity_history, trades, total_return, final_equity
    """
    price_data = fetch_historical_prices()
    all_dates  = price_data["dates"]
    n_days     = min(n_days, len(all_dates))
    dates      = all_dates[-n_days:]

    portfolio      = SimulatedPortfolio(initial_cash)
    equity_history = []
    all_trades     = []
    entry_indices  = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"BACKTEST  {dates[0]}  →  {dates[-1]}  ({len(dates)} days)")
        print(f"Initial cash: ${initial_cash:,.0f}  |  Universe: {len(WHITELIST)} stocks")
        print(f"Agent: up to 6 tool-call turns per day")
        print(f"{'='*60}")

    for date in dates:
        day_prices = {
            sym: price_data["prices"].get(sym, {}).get(date)
            for sym in WHITELIST
        }
        day_prices = {k: v for k, v in day_prices.items() if v is not None}
        if len(day_prices) < 5:
            continue

        print(f"{date}:")
        # Agent decides; get_news tool calls are cached by date
        try:
            ts = int(datetime.datetime.strptime(date, "%Y-%m-%d").timestamp())
        except Exception:
            ts = int(time.time())

        state         = portfolio.get_state(day_prices, timestamp=ts)
        equity_before = state["equity"]

        decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
            tok, model, state, universe=WHITELIST, date=date
        )

        # Simulate trades
        day_trades = 0
        for d in decisions:
            result = portfolio.execute(d, day_prices)
            if result:
                all_trades.append({**result, "date": date})
                day_trades += 1

        equity_after = portfolio.equity(day_prices)
        equity_history.append({"date": date, "equity": round(equity_after, 2)})

        # Log episode
        if collector is not None:
            idx = collector.log(sft_prompt, sft_completion, decisions, equity_before, tool_calls_made)
            entry_indices.append(idx)
            if len(entry_indices) > 1:
                collector.update_outcome(entry_indices[-2], equity_after)

        pnl = equity_after - initial_cash
        if verbose:
            print(f"  Equity: ${equity_after:>10,.2f}  |  "
                  f"P&L: ${pnl:>+9,.2f}  |  Trades: {day_trades}")

    # Update last entry outcome
    if collector and entry_indices:
        final_eq = equity_history[-1]["equity"] if equity_history else initial_cash
        collector.update_outcome(entry_indices[-1], final_eq)

    final_equity = equity_history[-1]["equity"] if equity_history else initial_cash
    total_return = (final_equity - initial_cash) / initial_cash * 100

    if verbose:
        print(f"\n{'='*60}")
        print(f"RESULTS")
        print(f"  Period:  {dates[0]}  →  {dates[-1]}")
        print(f"  Initial: ${initial_cash:>12,.2f}")
        print(f"  Final:   ${final_equity:>12,.2f}")
        print(f"  Return:  {total_return:>+11.2f}%")
        print(f"  Trades:  {len(all_trades):>12d}")
        print(f"{'='*60}")

    return {
        "equity_history": equity_history,
        "trades":         all_trades,
        "total_return":   total_return,
        "final_equity":   final_equity,
    }


print("Backtesting infrastructure loaded.")


Backtesting infrastructure loaded.


### 1.15 Test Agent Decision Engine

Runs a quick smoke test of `llm_agent_decide()` using only the tools that are
currently implemented (i.e. registered with a non-None `func`).

**At this point (before Task 2):**
- `get_news` is available — the agent can fetch financial news
- `get_price_history`, `get_rsi`, `get_macd`, `get_global_quote` are not yet available

After completing Task 2 and registering your tools, re-run this cell to see
the agent use all of your implementations.


In [ ]:
# Quick smoke test — only tools implemented so far will be available
print("Available tools before Task 2:")
list_tools()

print("\nRunning agent on 5-symbol subset...")
state = get_state(tc)
decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
    tok, model, state, universe=WHITELIST[:5]
)

print(f"\nDecisions ({len(decisions)}):")
for d in decisions:
    sym    = d.get("symbol", "?")
    act    = d["action"].upper()
    qty    = d.get("qty", "-")
    reason = d.get("reason", "")[:60]
    print(f"  [{sym}] {act:5s}  qty={str(qty):>4}  {reason}")

print(f"\nSFT prompt  : {len(sft_prompt):,} chars")
print(f"SFT completion: {len(sft_completion):,} chars")


Available tools before Task 2:
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news + sentiment for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    OK             Get recent daily OHLCV prices for a stock via yfinance
  get_rsi              OK             Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             OK             Get MACD trend indicator via Alpha Vantage
  get_global_quote     OK             Get real-time price quote via Alpha Vantage
  search_market_news   OK             Discover trending stocks from broad market news by topic. Returns top mentioned whitelist tickers with sentiment and headlines.
  get_earnings         OK             Get latest quarterly earnings and EPS

### Task 1 Complete!

You now have all core infrastructure running:
- LLM model loaded
- Alpaca trading client connected
- News cached to Google Drive
- Decision engine ready
- Training data collector ready (for Task 4)
- Backtesting engine ready (for Task 4)

**Next: Implement Tasks 2-5 below.**

---

## Task 2: [TODO] Implement MCP Tool Functions

### Objective

Implement the **data retrieval tools** that the trading agent uses to research stocks
before making decisions. When the agent outputs a tool call, `dispatch_tool()` routes
it to your function — so the quality of your implementations directly determines
how well-informed the agent's decisions are.

### Tool Registry Reference

All tool functions you implement in this task must be registered using the `register_tool()`
function from **Section 1.10**. After you write each function, call:

```python
register_tool(
    name        = "your_tool_name",
    description = "brief description for the agent",
    params      = {"param": "type — description"},
    func        = your_function,
)
```

See Section 1.10 for the complete Tool Registry API documentation.

### How a tool call reaches your code

```
Agent outputs:
  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA", "time_period": 14}}</tool_call>

dispatch_tool("get_rsi", {"symbol": "NVDA", "time_period": 14})
  └─ _tool_registry["get_rsi"]["func"]("NVDA", time_period=14)
       └─ YOUR get_rsi("NVDA", time_period=14)   ← implement this
            └─ returns: "RSI(14) for NVDA: 62.3 (neutral) [2024-01-15]"
```

### [TODO] Required: implement and register these tools

**Implementation order:** Start with `get_news_for_symbols` (Section 1.9) as it demonstrates the complete pattern: API call → error handling → caching → formatting. Then implement the remaining functions following the same pattern.

| Function | Data source | What to implement |
|----------|------------|-------------------|
| `get_news_for_symbols(symbols, limit, date)` | **Alpha Vantage** | News + sentiment with caching |
| `get_price_history(symbol, period)` | **yfinance** | Recent OHLCV table |
| `get_rsi(symbol, time_period)` | **Alpha Vantage** | RSI value + signal |
| `get_macd(symbol)` | **Alpha Vantage** | MACD, signal, histogram |
| `get_global_quote(symbol)` | **Alpha Vantage** | Live price, change, volume |

After implementing each function you **must** call `register_tool()` to update the
registry — otherwise the agent won't know the tool is ready.

### Alpha Vantage quick reference

```python
# RSI
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "RSI", "symbol": symbol,
    "interval": "daily", "time_period": time_period,
    "series_type": "close", "apikey": ALPHA_VANTAGE_API_KEY,
})
data = resp.json()
# data["Technical Analysis: RSI"] -> {date: {"RSI": "65.32"}, ...}

# MACD
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "MACD", "symbol": symbol,
    "interval": "daily", "series_type": "close",
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Technical Analysis: MACD"] -> {date: {"MACD":..,"MACD_Signal":..,"MACD_Hist":..}}

# Global Quote
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "GLOBAL_QUOTE", "symbol": symbol,
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Global Quote"] -> {"05. price": "182.63", "09. change": "+1.20", ...}
```

### yfinance quick reference

```python
import yfinance as yf
ticker = yf.Ticker("AAPL")
hist   = ticker.history(period="5d")   # pandas DataFrame
# columns: Open, High, Low, Close, Volume
# index:   DatetimeIndex
```

### Optional: add your own custom tool

Implement any additional data source (earnings, options flow, sector ETF performance,
macroeconomic indicators, etc.) and register it so the agent can call it automatically:

```python
def get_earnings(symbol: str) -> str:
    """Fetch latest quarterly earnings from Alpha Vantage."""
    # function="EARNINGS" -> data["quarterlyEarnings"][0]
    ...

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# Agent now calls get_earnings automatically when it wants earnings data
```

### Grading criteria

- [ ] `get_news_for_symbols` implements full caching + API call pattern
- [ ] `get_price_history` returns formatted OHLCV string
- [ ] At least **2** Alpha Vantage functions implemented
- [ ] Each function handles API errors gracefully (try/except)
- [ ] All implemented functions registered with `register_tool(..., func=your_fn)`
- [ ] (Optional) custom tools added and registered


In [ ]:
# =============================================================================
# TODO: Task 2 — Implement MCP Tool Functions
# Each function is called by dispatch_tool() when the agent requests it.
# After implementing a function, update its registry entry so the agent can use it.
# =============================================================================

import requests
import yfinance as yf
from collections import defaultdict

def _clean_symbol(symbol):
    """Normalize symbol input and reject obviously bad values."""
    if isinstance(symbol, list):
        if len(symbol) == 1:
            symbol = symbol[0]
        else:
            return None

    if symbol is None:
        return None

    symbol = str(symbol).upper().strip()
    return symbol if symbol else None


def _short_alpha_vantage_note(tool_name: str, symbol: str = None) -> str:
    base = f"{tool_name} unavailable"
    if symbol:
        base += f" for {symbol}"
    return base + " due to API rate limit."


# ── Required: get_price_history ───────────────────────────────────────────────


def get_price_history(symbol: str, period: str = "5d") -> str:
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period)

        if hist.empty:
            return f"No price history available for {symbol}."

        lines = [f"Price history for {symbol} ({period}):"]

        for date, row in hist.iterrows():
            d = date.strftime("%Y-%m-%d")
            o = row["Open"]
            h = row["High"]
            l = row["Low"]
            c = row["Close"]
            v = int(row["Volume"])

            lines.append(
                f"  {d}: O={o:.2f} H={h:.2f} L={l:.2f} C={c:.2f} V={v}"
            )

        return "\n".join(lines)

    except Exception as e:
        return f"Price history fetch failed for {symbol}: {e}"


register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = get_price_history,   # update func here after implementing above
)


# ── Required: get_rsi ─────────────────────────────────────────────────────────

def get_rsi(symbol: str, interval: str = "daily", time_period: int = 14) -> str:
    """
    Fetch RSI (Relative Strength Index) for a symbol from Alpha Vantage.

    RSI > 70  →  overbought (potential sell signal)
    RSI < 30  →  oversold   (potential buy signal)
    30–70     →  neutral

    Alpha Vantage params:
        function="RSI", symbol, interval="daily",
        time_period=14, series_type="close", apikey

    Returns:
        e.g. "RSI(14) for NVDA: 68.5 (neutral — approaching overbought) [2024-01-15]"
    """
    # TODO: implement
    symbol = _clean_symbol(symbol)
    if not symbol:
        return "Invalid symbol."

    params = {
        "function": "RSI",
        "symbol": symbol,
        "interval": interval,
        "time_period": time_period,
        "series_type": "close",
        "apikey": ALPHA_VANTAGE_API_KEY,
    }

    try:
        resp = requests.get(ALPHA_VANTAGE_BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"RSI fetch failed for {symbol}: {e}"

    if "Error Message" in data:
        return f"RSI API error for {symbol}."
    if "Note" in data or "Information" in data:
        return _short_alpha_vantage_note("RSI", symbol)

    ta = data.get("Technical Analysis: RSI")
    if not ta:
        return f"No RSI data available for {symbol}."

    try:
        latest_date = sorted(ta.keys())[-1]
        rsi_val = float(ta[latest_date]["RSI"])

        if rsi_val > 70:
            label = "overbought — potential sell signal"
        elif rsi_val < 30:
            label = "oversold — potential buy signal"
        elif rsi_val >= 60:
            label = "neutral — bullish momentum"
        elif rsi_val <= 40:
            label = "neutral — bearish momentum"
        else:
            label = "neutral"

        return f"RSI({time_period}) for {symbol}: {rsi_val:.2f} ({label}) [{latest_date}]"

    except Exception as e:
        return f"RSI parse failed for {symbol}: {e}"


register_tool(
    name="get_rsi",
    description="Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params={"symbol": "str — stock ticker", "time_period": "int — default 14"},
    func=get_rsi,
)


# ── Required: get_macd ────────────────────────────────────────────────────────

def get_macd(symbol: str, interval: str = "daily") -> str:
    """
    Fetch MACD (Moving Average Convergence Divergence) for a symbol.

    MACD > Signal  →  bullish crossover
    MACD < Signal  →  bearish crossover

    Alpha Vantage params:
        function="MACD", symbol, interval="daily",
        series_type="close", apikey

    Response key: data["Technical Analysis: MACD"]
        {date: {"MACD": "0.1234", "MACD_Signal": "0.0987", "MACD_Hist": "0.0247"}}

    Returns:
        e.g. "MACD for NVDA [2024-01-15]: MACD=0.1234, Signal=0.0987, Hist=0.0247 (bullish)"
    """
    # TODO: implement
    symbol = _clean_symbol(symbol)
    if not symbol:
        return "Invalid symbol."

    params = {
        "function": "MACD",
        "symbol": symbol,
        "interval": interval,
        "series_type": "close",
        "apikey": ALPHA_VANTAGE_API_KEY,
    }

    try:
        resp = requests.get(ALPHA_VANTAGE_BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"MACD fetch failed for {symbol}: {e}"

    if "Error Message" in data:
        return f"MACD API error for {symbol}."
    if "Note" in data or "Information" in data:
        return _short_alpha_vantage_note("MACD", symbol)

    ta = data.get("Technical Analysis: MACD")
    if not ta:
        return f"No MACD data available for {symbol}."

    try:
        latest_date = sorted(ta.keys())[-1]
        latest = ta[latest_date]

        macd_val = float(latest["MACD"])
        signal_val = float(latest["MACD_Signal"])
        hist_val = float(latest["MACD_Hist"])

        if macd_val > signal_val:
            trend = "bullish crossover"
        elif macd_val < signal_val:
            trend = "bearish crossover"
        else:
            trend = "neutral crossover"

        return (
            f"MACD for {symbol} [{latest_date}]: "
            f"MACD={macd_val:.4f}, Signal={signal_val:.4f}, "
            f"Hist={hist_val:.4f} ({trend})"
        )

    except Exception as e:
        return f"MACD parse failed for {symbol}: {e}"


register_tool(
    name="get_macd",
    description="Get MACD trend indicator via Alpha Vantage",
    params={"symbol": "str — stock ticker"},
    func=get_macd,
)



# ── Required: get_global_quote ────────────────────────────────────────────────

def get_global_quote(symbol: str) -> str:
    """
    Fetch real-time price quote for a symbol from Alpha Vantage.

    Alpha Vantage params:
        function="GLOBAL_QUOTE", symbol, apikey

    Response key: data["Global Quote"]
        "05. price", "09. change", "10. change percent", "06. volume"

    Returns:
        e.g. "Quote AAPL: $186.10 | change: +1.20 (+0.65%) | vol: 52,341,200"
    """
    # TODO: implement
    symbol = _clean_symbol(symbol)
    if not symbol:
        return "Invalid symbol."

    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": symbol,
        "apikey": ALPHA_VANTAGE_API_KEY,
    }

    try:
        resp = requests.get(ALPHA_VANTAGE_BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"Quote fetch failed for {symbol}: {e}"

    if "Error Message" in data:
        return f"Quote API error for {symbol}."
    if "Note" in data or "Information" in data:
        return _short_alpha_vantage_note("Quote", symbol)

    quote = data.get("Global Quote")
    if not quote:
        return f"No real-time quote available for {symbol}."

    try:
        price = float(quote.get("05. price", 0))
        change = quote.get("09. change", "N/A")
        change_pct = quote.get("10. change percent", "N/A")
        volume = quote.get("06. volume", "N/A")
        latest_day = quote.get("07. latest trading day", "Unknown date")

        try:
            volume_fmt = f"{int(volume):,}"
        except Exception:
            volume_fmt = str(volume)

        return (
            f"Quote {symbol} [{latest_day}]: ${price:.2f} | "
            f"change: {change} ({change_pct}) | vol: {volume_fmt}"
        )

    except Exception as e:
        return f"Quote parse failed for {symbol}: {e}"


register_tool(
    name="get_global_quote",
    description="Get real-time price quote via Alpha Vantage",
    params={"symbol": "str — stock ticker"},
    func=get_global_quote,
)


# =============================================================================
# =============================================================================
# Optional C: search_market_news -- agent-driven stock discovery
# =============================================================================
# Instead of researching pre-defined tickers, this tool fetches broad topic-based
# news from Alpha Vantage WITHOUT specifying any symbols. The agent reads the
# headlines and discovers interesting companies on its own.
#
# Alpha Vantage NEWS_SENTIMENT endpoint (no tickers= argument):
#   params: function="NEWS_SENTIMENT", topics=topic, limit=20, apikey=...
#   response: data["feed"] -> list of articles, each with "ticker_sentiment":
#     [{"ticker": "NVDA", "ticker_sentiment_score": "0.35", ...}, ...]
#
# def search_market_news(topic: str = "technology") -> str:
#     """
#     Fetch broad market news by topic; return top mentioned tickers + sentiment.
#
#     Args:
#         topic : one of "technology", "earnings", "ipo",
#                 "mergers_and_acquisitions", "financial_markets",
#                 "energy_transportation", "finance"
#     Returns:
#         Formatted string, e.g.:
#         "Top mentioned tickers for topic=technology:\n"
#         "  NVDA: avg_sentiment=0.42 (5 articles)\n"
#         "  MSFT: avg_sentiment=0.31 (3 articles)\n"
#         "Headlines:\n  - Nvidia announces next-gen GPU ...\n"
#     """
#     url = ALPHA_VANTAGE_BASE_URL
#     params = {"function": "NEWS_SENTIMENT", "topics": topic,
#               "limit": 20, "apikey": ALPHA_VANTAGE_API_KEY}
#     # TODO: call API, parse feed, aggregate ticker_sentiment by symbol
#     raise NotImplementedError("TODO: implement search_market_news()")
#
# register_tool(
#     name        = "search_market_news",
#     description = (
#         "Fetch broad market news by topic (no ticker needed). "
#         "Topics: technology, earnings, ipo, mergers_and_acquisitions, "
#         "financial_markets, energy_transportation, finance."
#     ),
#     params      = {"topic": "str"},
#     func        = search_market_news,
# )

# Optional: add your own custom tool
# =============================================================================
#
# def get_MY_tool(symbol: str) -> str:
#     """Your description here."""
#     # ... implementation ...
#     return formatted_string
#
# register_tool(
#     name        = "get_MY_tool",
#     description = "One-line description for the agent",
#     params      = {"symbol": "str — stock ticker"},
#     func        = get_MY_tool,
# )


# =============================================================================
# get_extra_context
# =============================================================================
# This function aggregates data from your MCP tools into a single context string
# that gets passed to llm_agent_decide() in the trading loop (Task 5, Step 3).
#
# It calls get_daily_prices, get_rsi, get_macd, and get_global_quote for each
# symbol, collects the results, and joins them into one big string.
# If you implemented Optional C (search_market_news), call it here too.
# =============================================================================

def search_market_news(topic: str = "technology") -> str:
    """
    Fetch broad market news by topic; return top mentioned tickers + sentiment.
    """

    try:
        resp = requests.get(
            ALPHA_VANTAGE_BASE_URL,
            params={
                "function": "NEWS_SENTIMENT",
                "topics": topic,
                "limit": 20,
                "apikey": ALPHA_VANTAGE_API_KEY,
            },
            timeout=20,
        )
        resp.raise_for_status()
        data = resp.json()

        if "feed" not in data:
            if "Note" in data or "Information" in data:
                return _short_alpha_vantage_note("Market news")
            return f"Market news fetch failed for topic={topic}."

        feed = data["feed"]
        if not feed:
            return f"No market news found for topic={topic}."

        ticker_scores = defaultdict(list)
        headlines = []

        for article in feed:
            title = article.get("title", "No title")
            headlines.append(title)

            for item in article.get("ticker_sentiment", []):
                ticker = item.get("ticker")
                score = item.get("ticker_sentiment_score")

                if ticker is None or score is None:
                    continue
                if ticker not in WHITELIST:
                    continue

                try:
                    ticker_scores[ticker].append(float(score))
                except Exception:
                    pass

        lines = [f"=== Market Discovery: topic={topic} ==="]

        if ticker_scores:
            ranked = sorted(
                ticker_scores.items(),
                key=lambda kv: (len(kv[1]), sum(kv[1]) / len(kv[1])),
                reverse=True,
            )[:5]

            lines.append("Top tickers (by mentions + sentiment):")
            for ticker, scores in ranked:
                avg_sentiment = sum(scores) / len(scores)
                lines.append(
                    f"  {ticker}: avg_sentiment={avg_sentiment:.2f} ({len(scores)} mentions)"
                )
        else:
            lines.append("No relevant whitelist tickers found in news.")

        lines.append("")
        lines.append("Sample headlines:")
        for title in headlines[:5]:
            lines.append(f"  - {title}")

        return "\n".join(lines)

    except Exception as e:
        return f"Market news fetch failed for topic={topic}: {e}"


register_tool(
    name="search_market_news",
    description=(
        "Discover trending stocks from broad market news by topic. "
        "Returns top mentioned whitelist tickers with sentiment and headlines."
    ),
    params={"topic": "str — e.g. technology, earnings, finance"},
    func=search_market_news,
)

def get_earnings(symbol: str) -> str:
    """
    Fetch latest quarterly earnings from Alpha Vantage.
    """
    symbol = _clean_symbol(symbol)
    if not symbol:
        return "Invalid symbol."

    try:
        resp = requests.get(
            ALPHA_VANTAGE_BASE_URL,
            params={
                "function": "EARNINGS",
                "symbol": symbol,
                "apikey": ALPHA_VANTAGE_API_KEY,
            },
            timeout=20,
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"Earnings fetch failed for {symbol}: {e}"

    if "quarterlyEarnings" not in data or not data["quarterlyEarnings"]:
        if "Note" in data or "Information" in data:
            return _short_alpha_vantage_note("Earnings", symbol)
        return f"No earnings data available for {symbol}."

    try:
        q = data["quarterlyEarnings"][0]

        reported_date = q.get("reportedDate", "N/A")
        reported_eps = q.get("reportedEPS", "N/A")
        estimated_eps = q.get("estimatedEPS", "N/A")
        surprise = q.get("surprise", "N/A")
        surprise_pct = q.get("surprisePercentage", "N/A")

        return (
            f"Earnings for {symbol}: reportedDate={reported_date}, "
            f"reportedEPS={reported_eps}, estimatedEPS={estimated_eps}, "
            f"surprise={surprise}, surprisePct={surprise_pct}"
        )

    except Exception as e:
        return f"Earnings parse failed for {symbol}: {e}"


register_tool(
    name="get_earnings",
    description="Get latest quarterly earnings and EPS surprise for a stock",
    params={"symbol": "str — stock ticker"},
    func=get_earnings,
)

def get_extra_context(symbols: list) -> str:

    """
    Aggregate outputs from all implemented MCP tools for the given symbols.

    For each symbol in the list, call:
      - get_daily_prices(symbol)
      - get_rsi(symbol)
      - get_macd(symbol)
      - get_global_quote(symbol)

    Collect results into a single formatted string. Each tool call should be
    wrapped in try/except so one failure doesn't block the rest.

    If you implemented search_market_news (Bonus C), also call it with a
    relevant topic (e.g. "technology") and append the result.

    Args:
        symbols : list of ticker strings, e.g. ["AAPL", "NVDA"]

    Returns:
        Combined string of all tool outputs, separated by newlines.
        Returns "" if all calls fail.

    Example output:
        === Extra Context for AAPL ===
        Price history for AAPL (5d):
          2024-01-15: O=185.00 H=186.50 L=184.20 C=186.10 V=52341200
          ...
        RSI(14) for AAPL: 68.5 (neutral)
        MACD for AAPL: MACD=0.12, Signal=0.10, Hist=0.02 (bullish)
        Quote AAPL: 86.10 | change: +1.20 (+0.65%) | vol: 52,341,200
        ============================================

    Hints:
        tools = [get_daily_prices, get_rsi, get_macd, get_global_quote]
        for symbol in symbols:
            for tool in tools:
                try: results.append(tool(symbol))
                except: pass
    """
    # TODO: implement

    if not symbols:
        return ""

    results = []

    for symbol in symbols:
        symbol = _clean_symbol(symbol)
        if not symbol:
            continue

        section = [f"=== Extra Context for {symbol} ==="]

        tools = [
            lambda s: get_price_history(s, period="5d"),
            get_rsi,
            get_macd,
            get_global_quote,
            get_earnings,
        ]

        for tool in tools:
            try:
                out = tool(symbol)
                if out:
                    section.append(out)
            except Exception as e:
                section.append(f"Tool failed for {symbol}: {e}")

        section.append("=" * 44)
        results.append("\n".join(section))

    try:
        results.append(search_market_news("technology"))
    except Exception:
        pass

    return "\n\n".join(results)

### Test Task 2

In [ ]:
# Test your implementations
print("Testing MCP Tool Functions...")
print("=" * 60)

for func_name, func in [("get_rsi", get_rsi), ("get_macd", get_macd),
                          ("get_price_history", get_price_history),
                          ("get_global_quote", get_global_quote)]:
    try:
        result = func("AAPL")
        print(f"{func_name}('AAPL'):")
        print(f"  {result[:200]}")
    except NotImplementedError:
        print(f"{func_name}: Not yet implemented")
    except Exception as e:
        print(f"{func_name}: Error - {e}")

print("" + "=" * 60)

# Test get_extra_context (aggregates all tool outputs)
try:
    extra = get_extra_context(["AAPL"])
    assert isinstance(extra, str), f"Expected str, got {type(extra)}"
    print(f"get_extra_context(['AAPL']):")
    print(f"  Length: {len(extra)} chars")
    print(f"  Preview: {extra[:300]}")
except NotImplementedError:
    print("get_extra_context: Not yet implemented")
except Exception as e:
    print(f"get_extra_context: Error - {e}")

# Test that get_extra_context handles multiple symbols
try:
    extra_multi = get_extra_context(["AAPL", "NVDA"])
    assert isinstance(extra_multi, str), f"Expected str, got {type(extra_multi)}"
    assert len(extra_multi) >= len(extra), "Multi-symbol result should be >= single-symbol"
    print(f"get_extra_context(['AAPL', 'NVDA']):")
    print(f"  Length: {len(extra_multi)} chars")
except NotImplementedError:
    pass
except Exception as e:
    print(f"get_extra_context multi-symbol: Error - {e}")

print("" + "=" * 60)
print("Task 2 testing complete.")


Testing MCP Tool Functions...
get_rsi('AAPL'):
  RSI unavailable for AAPL due to API rate limit.
get_macd('AAPL'):
  MACD unavailable for AAPL due to API rate limit.
get_price_history('AAPL'):
  Price history for AAPL (5d):
  2026-04-08: O=258.45 H=259.75 L=256.53 C=258.90 V=41032800
  2026-04-09: O=259.00 H=261.12 L=256.07 C=260.49 V=28121600
  2026-04-10: O=259.98 H=262.19 L=259.02 C=260.48
get_global_quote('AAPL'):
  Quote unavailable for AAPL due to API rate limit.
get_extra_context(['AAPL']):
  Length: 657 chars
  Preview: === Extra Context for AAPL ===
Price history for AAPL (5d):
  2026-04-08: O=258.45 H=259.75 L=256.53 C=258.90 V=41032800
  2026-04-09: O=259.00 H=261.12 L=256.07 C=260.49 V=28121600
  2026-04-10: O=259.98 H=262.19 L=259.02 C=260.48 V=31291500
  2026-04-13: O=259.73 H=260.18 L=256.66 C=259.20 V=35196
get_extra_context(['AAPL', 'NVDA']):
  Length: 1272 chars
Task 2 testing complete.


---
## Task 3: [TODO] RAG Memory System

### Objective
Implement a memory system that allows the trading agent to **learn from past trades**.

Two components:
1. **`TradingMemory`** class - stores situation-reflection pairs, retrieves similar experiences using BM25
2. **`reflect_on_trade()`** function - uses LLM to generate a lesson learned after a trade

### How it works

```
Trade executed -> Wait N loops -> Compare equity -> LLM reflection -> Store in memory
                                                                           |
Next decision <- Retrieve similar past reflections via BM25 <--------------+
```

### BM25 Retrieval
[BM25](https://en.wikipedia.org/wiki/Okapi_BM25) is a keyword-based ranking function:
- Scores documents based on **term frequency** and **inverse document frequency**
- The `rank_bm25` library provides `BM25Okapi`

```python
from rank_bm25 import BM25Okapi

corpus = ["apple stock rises today", "tesla earnings beat expectations"]
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)
scores = bm25.get_scores("apple earnings".lower().split())
# scores = [0.45, 0.32]  -- first doc scores higher for "apple"
```

### Grading Criteria
- TradingMemory correctly stores and loads memories
- BM25 retrieval returns relevant past reflections
- reflect_on_trade generates meaningful short reflections
- Edge cases handled (empty memory, no matching results)


### 3a. TradingMemory Class

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
class TradingMemory:
    """
    BM25-based memory system for storing and retrieving trading reflections.
    Each memory entry: {"situation": str, "reflection": str, "timestamp": int}
    """

    def __init__(self, memory_file: str = "trading_memory.json"):
        """
        Initialize the memory system.

        TODO:
        - Store memory_file path as self.memory_file
        - Initialize self.memories as empty list
        - Load existing memories from file if it exists
          (use json.load, handle file-not-found and parse errors gracefully)
        """
        # TODO: Implement
        if memory_file is None:
            memory_file = os.path.join(CACHE_DIR, "trading_memory.json")

        self.memory_file = memory_file
        self.memories = []

        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if isinstance(data, list):
                    self.memories = data
                else:
                    print("Memory file format invalid, starting fresh.")
                    self.memories = []

                print(f"Loaded {len(self.memories)} trading memories")

            except Exception as e:
                print(f"Memory file corrupted, starting fresh: {e}")
                self.memories = []
    def _save(self):
        """Persist memories to disk as JSON."""
        with open(self.memory_file, "w", encoding="utf-8") as f:
            json.dump(self.memories, f, ensure_ascii=False, indent=2)

    def add_memory(self, situation: str, reflection: str):
        """
        Store a new situation-reflection pair.

        TODO:
        - Append {"situation": situation, "reflection": reflection,
                   "timestamp": int(time.time())} to self.memories
        - Call self._save()
        """
        # TODO: Implement
        entry = {
            "situation": situation.strip() if isinstance(situation, str) else "",
            "reflection": reflection.strip() if isinstance(reflection, str) else "",
            "timestamp": int(time.time())
        }

        self.memories.append(entry)
        self._save()

    def get_memories(self, query: str, n_matches: int = 2) -> list:
      try:
            if not self.memories:
                return []

            query = (query or "").strip()
            if not query:
                return []

            corpus = [m.get("situation", "") for m in self.memories]
            tokenized_corpus = [doc.lower().split() for doc in corpus]

            tokenized_query = query.lower().split()
            if not tokenized_query:
                return []

            bm25 = BM25Okapi(tokenized_corpus)
            scores = bm25.get_scores(tokenized_query)

            ranked_indices = sorted(
                range(len(scores)),
                key=lambda i: scores[i],
                reverse=True
            )[:n_matches]

            results = []
            for idx in ranked_indices:
                if scores[idx] > 0:
                    reflection = self.memories[idx].get("reflection", "")
                    if reflection:
                        results.append(reflection)

            return results

      except Exception as e:
          print(f"Memory retrieval failed: {e}")
          return []

### 3b. Trade Reflection Function

In [ ]:
def reflect_on_trade(tok, model, situation: str, decision: dict,
                     prev_equity: float, curr_equity: float) -> str:
    """
    Use LLM to generate a short reflection on a trade outcome.

    Args:
        tok: Tokenizer
        model: LLM model
        situation: Market situation when decision was made
        decision: The trading decision dict
        prev_equity: Portfolio equity before the trade
        curr_equity: Portfolio equity after the trade

    Returns:
        A short reflection string (max 300 chars).
        Return "No reflection generated." if generation fails.

    TODO:
    1. Calculate P&L: pnl = curr_equity - prev_equity
       Calculate P&L pct: pnl_pct = (pnl / prev_equity) * 100
    2. Build a prompt asking the LLM to write a 1-2 sentence lesson learned:
       - System: "You are a trading reflection assistant..."
       - User: Include the decision (JSON), profit/loss amount, situation summary
       - Use prompt format: <|system|>...\n<|user|>...\n<|assistant|>\n
    3. Generate with max_new_tokens=100, do_sample=False
    4. Extract the assistant response, truncate to 300 chars
    """
    # TODO: Implement
    try:
        pnl = curr_equity - prev_equity
        pnl_pct = (pnl / prev_equity) * 100 if prev_equity != 0 else 0.0

        prompt = (
            "<|system|>\n"
            "You are a trading reflection assistant. "
            "Write a short, practical 1-2 sentence lesson learned from the trade outcome.\n"
            "<|user|>\n"
            f"Situation:\n{situation}\n\n"
            f"Decision:\n{json.dumps(decision)}\n\n"
            f"Previous equity: {prev_equity:.2f}\n"
            f"Current equity: {curr_equity:.2f}\n"
            f"P&L: {pnl:.2f} ({pnl_pct:.2f}%)\n\n"
            "Write a concise reflection about what was learned from this trade."
            "\n<|assistant|>\n"
        )

        inputs = tok(prompt, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

        generated_text = tok.decode(outputs[0], skip_special_tokens=False)

        if "<|assistant|>" in generated_text:
            reflection = generated_text.split("<|assistant|>")[-1].strip()
        else:
            reflection = generated_text.strip()

        reflection = reflection.replace("</s>", "").strip()
        reflection = reflection[:300]

        if not reflection:
            return "No reflection generated."

        return reflection

    except Exception:
        return "No reflection generated."

### Test Task 3

In [ ]:
# Test TradingMemory
try:
    test_mem = TradingMemory("test_memory.json")

    test_mem.add_memory(
        situation="AAPL bullish news, strong earnings, bought 10 shares",
        reflection="Buying on strong earnings news was profitable."
    )
    test_mem.add_memory(
        situation="TSLA bearish sentiment, missed earnings, sold 5 shares",
        reflection="Selling before bearish news helps protect capital."
    )
    test_mem.add_memory(
        situation="MSFT neutral news, market sideways, held position",
        reflection="Holding during uncertain times avoided transaction costs."
    )

    results = test_mem.get_memories("AAPL earnings report bullish", n_matches=2)
    print("Memory retrieval test:")
    print(f"  Query: 'AAPL earnings report bullish'")
    print(f"  Retrieved {len(results)} memories:")
    for i, r in enumerate(results, 1):
        print(f"    {i}. {r}")

    if os.path.exists("test_memory.json"):
        os.remove("test_memory.json")
    print("\nTradingMemory works!")
except NotImplementedError:
    print("Task 3a not yet implemented")
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()

# Test reflect_on_trade
try:
    reflection = reflect_on_trade(
        tok, model,
        situation="AAPL bullish news, bought 5 shares at $280",
        decision={"action": "buy", "symbol": "AAPL", "qty": 5},
        prev_equity=100000.0, curr_equity=100150.0
    )
    print(f"\nReflection test: {reflection}")
    print("reflect_on_trade works!")
except NotImplementedError:
    print("\nTask 3b (reflect_on_trade) not yet implemented")
except Exception as e:
    print(f"\nError: {e}")

Memory retrieval test:
  Query: 'AAPL earnings report bullish'
  Retrieved 1 memories:
    1. Buying on strong earnings news was profitable.

TradingMemory works!

Reflection test: From this trade, I learned that timely and informed buying can lead to significant gains in the stock market. However, it's crucial to monitor the market closely and adjust strategies as necessary to avoid potential losses. Always keep your risk management in mind when making trades. <|<|endoftext|>
reflect_on_trade works!


---
## Task 4: [TODO] SFT Fine-tuning with LoRA

### Objective
Fine-tune the Qwen 1.5B model on **backtested trading data** using LoRA (Low-Rank Adaptation), then compare performance.

### Flow

```
Step 1: Backtest with BASE model
         Simulate 20 trading days on historical prices
         LLM makes decisions each day (no real money)
         All (prompt, decisions, equity_change) logged automatically
                         â”‚
Step 2: Auto-label data
         equity went UP   â†’ label "good"
         equity went DOWN â†’ label "bad"
                         â”‚
Step 3: SFT fine-tune (YOUR IMPLEMENTATION)
         Filter "good" examples
         Tokenize as (prompt, completion) pairs
         LoRA: only train ~0.5% of parameters
         Save adapter to Drive
                         â”‚
Step 4: Backtest with FINE-TUNED model
         Same historical data, same starting cash
         Compare returns: base vs fine-tuned
```

### What you need to do

1. **`prepare_sft_dataset()`** - Filter labeled data, tokenize into training format
2. **`get_lora_model()`** - Configure LoRA adapters on the base model
3. **`train_sft()`** - Run the training loop
4. Compare backtest results

### Key Concepts

**LoRA** adds small trainable matrices to attention layers while freezing the base model:
- `r=8`: rank of the low-rank matrices (higher = more capacity but slower)
- `lora_alpha=16`: scaling factor (typically 2x r)
- `target_modules=["q_proj", "v_proj"]`: which layers to adapt
- Only ~0.5% of parameters are trainable â†’ fast training on Colab

**SFT (Supervised Fine-Tuning)**: Train the model to reproduce "good" trading decisions.
The loss is only computed on the **completion** (decisions), not the prompt.

### Grading Criteria
- Backtest runs successfully with base model
- `prepare_sft_dataset` correctly filters and tokenizes data
- LoRA config is reasonable (r, alpha, target modules)
- Training loop runs without errors
- Comparison backtest shows fine-tuned model produces valid decisions

### 4a. Run Backtest & Collect Training Data

Run a backtest with the **base model** on historical data. This collects trading decisions and outcomes
without spending any paper money. Data is auto-labeled based on P&L.

In [ ]:
# =============================================================================
# Run backtest with BASE model to collect training data
# =============================================================================

# Initialize a fresh collector (clears previous data)
import os as _os
_sft_path = _os.path.join(CACHE_DIR, "sft_data.jsonl")
if _os.path.exists(_sft_path):
    _os.remove(_sft_path)
    print("Cleared previous training data")

collector = TradingDataCollector()

# Run backtest - this takes a few minutes
print("Running backtest with BASE model...")
print("(Each day = 1 LLM inference, ~15-30 sec on T4)\n")
base_results = run_backtest(tok, model, collector=collector, n_days=80)

# Auto-label: good if equity increased, bad if decreased
collector.auto_label(threshold=20.0)

print("\n--- Training Data Summary ---")
collector.summary()

# Review labeled entries
print("\nRecent entries:")
collector.review(n=10)

# Manual override example (uncomment to use):
# collector.set_label(0, "good")   # override entry 0
# collector.set_label(3, "bad")    # override entry 3

print(f"\nGood examples for training: {len(collector.get_labeled('good'))}")
print(f"Bad examples (not used in SFT): {len(collector.get_labeled('bad'))}")

Cleared previous training data
Running backtest with BASE model...
(Each day = 1 LLM inference, ~15-30 sec on T4)

Loaded cached price data (61 trading days)

BACKTEST  2026-01-14  →  2026-04-13  (61 days)
Initial cash: $100,000  |  Universe: 100 stocks
Agent: up to 6 tool-call turns per day
2026-01-14:
[Agent turn 1] → forced search_market_news({'topic': 'technology'})
[Agent turn 2] → get_news({'symbols': ['AAPL', 'NVDA']})
[Agent] Finalising — 3 turn(s), 2 tool call(s)
[Agent] Finalising — 4 turn(s), 2 tool call(s)
  Equity: $100,000.00  |  P&L: $    +0.00  |  Trades: 1
2026-01-15:
[Agent turn 1] → forced search_market_news({'topic': 'technology'})
[Agent turn 2] → get_rsi({'symbol': 'AAPL', 'time_period': 14})
[Agent] Finalising — 3 turn(s), 2 tool call(s)
[Agent] Finalising — 4 turn(s), 2 tool call(s)
  Equity: $ 99,982.50  |  P&L: $   -17.50  |  Trades: 1
2026-01-16:
[Agent turn 1] → forced search_market_news({'topic': 'technology'})
[Agent turn 2] → get_news({'symbols': ['AAPL',

### Optional: Inspect & Edit SFT Training Data

After the backtest, `sft_data.jsonl` contains all logged (prompt, completion, label) pairs.
Use the cells below to review entries, remove low-quality ones, or upload a hand-edited version
before running `train_sft()`.


In [ ]:
# =============================================================================
# OPTIONAL: Inspect & Edit sft_data.jsonl
# Run each sub-block independently as needed.
# =============================================================================

import os
import json
from google.colab import files

sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

# -- 1. View all SFT entries -----------------------------------------------
if os.path.exists(sft_path):
    with open(sft_path, "r", encoding="utf-8") as f:
        entries = [json.loads(l) for l in f if l.strip()]

    print(f"Found {len(entries)} SFT entries:")
    good = [e for e in entries if e.get("label") == "good"]
    bad = [e for e in entries if e.get("label") == "bad"]
    print(f"  good: {len(good)}  |  bad: {len(bad)}\n")

    for i, e in enumerate(entries):
        label = e.get("label", "?")
        pnl = e.get("pnl", "?")
        compl = e.get("completion", "")[:100].replace("\n", " ")
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      completion: {compl}")
        print()
else:
    print("No sft_data.jsonl found yet. Run the backtest first.")

# -- 2. Download for local editing -----------------------------------------
# Uncomment only if you want to download the file
# if os.path.exists(sft_path):
#     files.download(sft_path)

# -- 3. Upload an edited version back --------------------------------------
# Uncomment only if you manually edited the file locally
# uploaded = files.upload()
# for fname, content in uploaded.items():
#     with open(sft_path, "wb") as f:
#         f.write(content)
# with open(sft_path, "r", encoding="utf-8") as f:
#     entries = [json.loads(l) for l in f if l.strip()]
# print(f"Reloaded {len(entries)} SFT entries from uploaded file.")

# -- 4. Keep only usable good entries for training -------------------------
if os.path.exists(sft_path):
    with open(sft_path, "r", encoding="utf-8") as f:
        entries = [json.loads(l) for l in f if l.strip()]

    print(f"\nBefore filtering: {len(entries)} entries")

    filtered = []

    for e in entries:
        if e.get("label") != "good":
            continue

        pnl = e.get("pnl", 0)
        completion = (e.get("completion") or "").strip()

        # keep decent wins, remove tiny/noisy ones
        if pnl is None or pnl < 30:
            continue

        # skip empty completions
        if not completion:
            continue

        filtered.append(e)

    print(f"After filtering: {len(filtered)} entries")

    # Only overwrite if we actually kept something
    if len(filtered) > 0:
        with open(sft_path, "w", encoding="utf-8") as f:
            for e in filtered:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")
        print(f"Kept {len(filtered)} usable examples; removed {len(entries) - len(filtered)} weak/bad ones.")
    else:
        print("Filtered dataset is empty — NOT overwriting file.")

# -- 5. Preview filtered file ----------------------------------------------
if os.path.exists(sft_path):
    with open(sft_path, "r", encoding="utf-8") as f:
        filtered_entries = [json.loads(l) for l in f if l.strip()]

    print(f"\nRemaining entries: {len(filtered_entries)}\n")

    for i, e in enumerate(filtered_entries[:10]):
        print(f"[{i}] pnl={e.get('pnl')} symbols={e.get('symbols')}")
        print(f"completion: {e.get('completion', '')[:200]}")
        print()

Found 38 SFT entries:
  good: 14  |  bad: 24

[ 0] label=bad  pnl=-17.5
      completion: [{"action": "buy", "symbol": "AAPL", "qty": 10, "order_type": "market", "limit_price": null, "reason

[ 1] label=bad  pnl=-29.479999999995925
      completion: [{"action": "buy", "symbol": "AAPL", "qty": 1, "order_type": "market", "limit_price": null, "reason"

[ 2] label=bad  pnl=-105.17999999999302
      completion: [{"action": "buy", "symbol": "NVDA", "qty": 1, "order_type": "market", "limit_price": null, "reason"

[ 3] label=bad  pnl=5.320000000006985
      completion: [{"action": "buy", "symbol": "MSFT", "qty": 1, "order_type": "market", "limit_price": null, "reason"

[ 4] label=bad  pnl=17.750000000014552
      completion: [{"action": "buy", "symbol": "NVDA", "qty": 1, "order_type": "market", "limit_price": null, "reason"

[ 5] label=bad  pnl=16.720000000001164
      completion: [{"action": "buy", "symbol": "AAPL", "qty": 1, "order_type": "market", "limit_price": null, "reason"

[ 6] label=g

In [ ]:
with open(sft_path) as f:
    entries = [json.loads(l) for l in f if l.strip()]

print(f"Remaining entries: {len(entries)}\n")

for i, e in enumerate(entries[:10]):
    print(f"[{i}] pnl={e.get('pnl')} symbols={e.get('symbols')}")
    print(f"completion: {e.get('completion', '')[:200]}")
    print()

Remaining entries: 8

[0] pnl=101.45000000001164 symbols=None
completion: [{"action": "buy", "symbol": "MSFT", "qty": 1, "order_type": "market", "limit_price": null, "reason": "small starter position after research"}]

[1] pnl=60.44999999999709 symbols=None
completion: [{"action": "buy", "symbol": "MSFT", "qty": 1, "order_type": "market", "limit_price": null, "reason": "small starter position after research"}]

[2] pnl=155.43999999998778 symbols=None
completion: [{"action": "buy", "symbol": "MSFT", "qty": 1, "order_type": "market", "limit_price": null, "reason": "small starter position after research"}]

[3] pnl=247.0799999999872 symbols=None
completion: [{"action": "buy", "symbol": "MSFT", "qty": 1, "order_type": "market", "limit_price": null, "reason": "High average sentiment and recent mentions suggest potential growth opportunities."}]

[4] pnl=216.08000000000175 symbols=None
completion: [{"action": "buy", "symbol": "MSFT", "qty": 1, "order_type": "market", "limit_price": null, "re

### 4b. Implement SFT Fine-tuning

In [36]:
from datasets import Dataset

In [37]:
from datasets import Dataset
import os, json, torch

def prepare_sft_dataset(tok, max_length=1024):
    sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

    if not os.path.exists(sft_path):
        raise ValueError("sft_data.jsonl not found")

    with open(sft_path, "r", encoding="utf-8") as f:
        entries = [json.loads(l) for l in f if l.strip()]

    if not entries:
        raise ValueError("No filtered SFT entries found")

    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for idx, entry in enumerate(entries):
        prompt = (entry.get("prompt") or "").strip()
        completion = (entry.get("completion") or "").strip()

        if not prompt or not completion:
            continue

        # tokenize separately
        prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        completion_ids = tok(completion, add_special_tokens=False)["input_ids"]

        # optional EOS
        if tok.eos_token_id is not None:
            completion_ids = completion_ids + [tok.eos_token_id]

        # reserve space so completion is not fully truncated away
        max_prompt_len = int(max_length * 0.7)
        max_completion_len = max_length - max_prompt_len

        prompt_ids = prompt_ids[:max_prompt_len]
        completion_ids = completion_ids[:max_completion_len]

        # skip if completion vanished
        if len(completion_ids) == 0:
            continue

        input_ids = prompt_ids + completion_ids
        attention_mask = [1] * len(input_ids)
        labels = [-100] * len(prompt_ids) + completion_ids[:]

        # final safety trim
        input_ids = input_ids[:max_length]
        attention_mask = attention_mask[:max_length]
        labels = labels[:max_length]

        # pad
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        pad_len = max_length - len(input_ids)

        input_ids += [pad_id] * pad_len
        attention_mask += [0] * pad_len
        labels += [-100] * pad_len

        # ensure some target tokens remain
        if all(x == -100 for x in labels):
            print(f"Skipping example {idx}: all labels masked")
            continue

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)
        all_labels.append(labels)

    if len(all_input_ids) == 0:
        raise ValueError("No valid training examples after masking/truncation")

    ds = Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    })

    print(f"SFT dataset prepared with {len(ds)} examples")
    return ds

### 4c. Re-run Backtest & Compare

Run the same backtest with the **fine-tuned model** and compare returns against the base model.

In [38]:
# =============================================================================
# Compare base vs fine-tuned model via backtest
# =============================================================================
print("=" * 60)
print("SFT Fine-tuning Pipeline")
print("=" * 60)

sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

if not os.path.exists(sft_path):
    print("\nNo sft_data.jsonl found!")
    print("Run the data collection / filtering step first.")
else:
    with open(sft_path, "r", encoding="utf-8") as f:
        filtered_entries = [json.loads(l) for l in f if l.strip()]

    good_count = len(filtered_entries)
    print(f"\nLoaded {good_count} existing training entries")

    if good_count == 0:
        print("\nNo filtered training data found!")
        print("Make sure you ran the filtering step first.")
    else:
        print(f"\nTraining on {good_count} examples...")

        try:
            # Step 1: Prepare dataset
            dataset = prepare_sft_dataset(tok)
            print(f"Dataset ready: {len(dataset)} examples")

            # Step 2: Create LoRA model
            lora_model = get_lora_model(model)

            # Step 3: Train
            lora_model = train_sft(lora_model, dataset, tok)
            print("\nTraining complete!")

            # Step 4: Backtest base model
            print("\n" + "=" * 60)
            print("BACKTEST: Base Model (comparison baseline)")
            print("=" * 60)
            base_results = run_backtest(tok, model, collector=None, n_days=20)

            # Step 5: Backtest fine-tuned model
            print("\n" + "=" * 60)
            print("BACKTEST: Fine-tuned Model")
            print("=" * 60)
            ft_results = run_backtest(tok, lora_model, collector=None, n_days=20)

            # Step 6: Compare
            print("\n" + "=" * 60)
            print("COMPARISON: Base Model vs Fine-tuned Model")
            print("=" * 60)
            print(f"  Base model return:       {base_results['total_return']:>+8.2f}%")
            print(f"  Fine-tuned model return: {ft_results['total_return']:>+8.2f}%")

            diff = ft_results['total_return'] - base_results['total_return']
            print(f"  Improvement:             {diff:>+8.2f}%")
            print("")
            print(f"  Base trades:             {len(base_results['trades']):>8d}")
            print(f"  Fine-tuned trades:       {len(ft_results['trades']):>8d}")
            print("=" * 60)

            if diff > 0:
                print("\nFine-tuned model outperformed the base model!")
            elif diff < 0:
                print("\nBase model performed better. Consider:")
                print("  - Collecting more training data")
                print("  - Filtering more carefully")
                print("  - Tuning LoRA hyperparameters")
            else:
                print("\nBoth models performed equally.")

        except NotImplementedError as e:
            print(f"\nNot yet implemented: {e}")
        except Exception as e:
            print(f"\nError: {e}")
            import traceback
            traceback.print_exc()

SFT Fine-tuning Pipeline

Loaded 13 existing training entries

Training on 13 examples...
SFT dataset prepared with 13 examples
Dataset ready: 13 examples
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Epoch 1/4 | Avg Loss: 1.8344
Epoch 2/4 | Avg Loss: 1.5870
Epoch 3/4 | Avg Loss: 1.3025
Epoch 4/4 | Avg Loss: 0.9954
LoRA adapter saved to /content/drive/MyDrive/stock_agent_cache/lora_adapter

Training complete!

BACKTEST: Base Model (comparison baseline)
Loaded cached price data (61 trading days)

BACKTEST  2026-03-16  →  2026-04-13  (20 days)
Initial cash: $100,000  |  Universe: 100 stocks
Agent: up to 6 tool-call turns per day
2026-03-16:
[Agent turn 1] → forced search_market_news({'topic': 'technology'})
[Agent] Finalising — 2 turn(s), 1 tool call(s)
[Agent turn 3] → get_news({'symbols': ['AAPL', 'NVDA']})
[Agent] Finalising — 4 turn(s), 2 tool call(s)
  Equity: $100,000.00  |  P&L: $    +0.00  |  Trades: 1
2026-03-17:
[Agent turn 1] → forced search_ma

---
## Task 5: [TODO] Full Trading Loop Integration

### Objective
Wire Tasks 2-3 together with the pre-written infrastructure into a complete trading loop, using the **fine-tuned model** from Task 4.

Each iteration:
1. Get current account state
2. Fetch financial news (from Drive cache)
3. Fetch extra context from your MCP tool functions (Task 2)
4. Check pending trades for reflection (after delay)
5. Retrieve similar past experiences from memory
6. Make LLM-powered decisions
7. Execute orders
8. Queue trades for future reflection

### Important Notes
- The loop runs for `n_iterations` (not infinite)
- Use `time.sleep(LOOP_SECONDS)` between iterations
- `pending_reflections` tracks trades awaiting reflection
- A trade is ready for reflection when `loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS`

### Grading Criteria
- All steps are correctly wired together
- Error handling prevents one failed step from crashing the loop
- `extra_context` from Task 2 is passed to `llm_agent_decide()`
- Memory from Task 3 is used for reflections and retrieval
- Orders are only submitted for buy/sell decisions (not hold)

In [39]:
# =============================================================================
# Task 5 — Final Trading Loop Integration
# =============================================================================

import time

def run_trading_loop(tc, tok, model, memory=None, collector=None, n_iterations=5):
    """
    Execute the main trading loop.

    Args:
        tc: Alpaca TradingClient
        tok: Tokenizer
        model: LLM model (prefer fine-tuned lora_model from Task 4)
        memory: TradingMemory instance (None to disable memory)
        collector: TradingDataCollector instance (None to disable data collection)
        n_iterations: Number of loop iterations to run
    """
    pending_reflections = []
    pending_outcomes = []

    total_tool_calls = 0
    total_real_trades = 0
    total_buys = 0
    total_sells = 0
    total_holds = 0

    for loop_count in range(1, n_iterations + 1):
        print(f"\n{'='*60}")
        print(f"LOOP #{loop_count} / {n_iterations} | Pending reflections: {len(pending_reflections)}")
        print("=" * 60)

        # --------------------------------------------------------------
        # Step 1: Get current account state
        # --------------------------------------------------------------
        try:
            state = get_state(tc)
        except Exception as e:
            print(f"[State Error] {e}")
            if loop_count < n_iterations:
                print(f"\nSleeping {LOOP_SECONDS} seconds before next loop...")
                time.sleep(LOOP_SECONDS)
            continue

        print("\nCurrent Account State:")
        print(f"Equity: ${state.get('equity', 0):,.2f}")
        print(f"Cash:   ${state.get('cash', 0):,.2f}")
        print(f"Positions: {len(state.get('positions', []))}")

        if state.get("positions"):
            for p in state["positions"]:
                symbol = p.get("symbol", "UNKNOWN")
                qty = p.get("qty", 0)
                market_value = p.get("market_value", 0)
                try:
                    market_value = float(market_value)
                except Exception:
                    market_value = 0.0
                print(f" - {symbol}: qty={qty}, value=${market_value:,.2f}")

        # --------------------------------------------------------------
        # Step 2: Fetch financial news
        # --------------------------------------------------------------
        try:
            news_context = get_news_for_symbols(WHITELIST)
            print("[News] Loaded")
        except Exception as e:
            print(f"[News Error] {e}")
            news_context = ""

        # --------------------------------------------------------------
        # Step 3: Fetch extra context from MCP tools
        # --------------------------------------------------------------
        try:
            held = [p["symbol"] for p in state.get("positions", [])][:3]

            if held:
                watchlist = held
            else:
                start_idx = (loop_count * 3) % len(WHITELIST)
                watchlist = WHITELIST[start_idx:start_idx + 3]
                if len(watchlist) < 3:
                    watchlist += WHITELIST[: 3 - len(watchlist)]

            extra_context = get_extra_context(watchlist)
            print(f"[Extra Context] Loaded for {watchlist}")
        except Exception as e:
            print(f"[Extra Context Error] {e}")
            extra_context = ""

        # --------------------------------------------------------------
        # Step 4: Check pending reflections
        # --------------------------------------------------------------
        still_pending_reflections = []

        for trade in pending_reflections:
            if loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS:
                try:
                    reflection = reflect_on_trade(
                        tok,
                        model,
                        trade["situation"],
                        trade["decision"],
                        trade["equity"],
                        state["equity"]
                    )

                    print(f"\n[Reflection] {trade['decision'].get('symbol', 'UNKNOWN')}:")
                    print(reflection)

                    if memory is not None and reflection:
                        memory.add_memory(trade["situation"], reflection)

                except Exception as e:
                    print(f"[Reflection Error] {e}")
                    still_pending_reflections.append(trade)
            else:
                still_pending_reflections.append(trade)

        pending_reflections = still_pending_reflections

        # --------------------------------------------------------------
        # Step 4.5: Update collector outcomes
        # --------------------------------------------------------------
        still_pending_outcomes = []

        for entry in pending_outcomes:
            if loop_count - entry["loop"] >= REFLECTION_DELAY_LOOPS:
                try:
                    if collector is not None:
                        collector.attach_outcome(entry["prompt"], state["equity"])
                except Exception as e:
                    print(f"[Collector Outcome Error] {e}")
                    still_pending_outcomes.append(entry)
            else:
                still_pending_outcomes.append(entry)

        pending_outcomes = still_pending_outcomes

        # --------------------------------------------------------------
        # Step 5: Retrieve past experiences from memory
        # --------------------------------------------------------------
        if memory is not None:
            situation_summary = f"""
Equity: {state.get('equity', 0)}
Cash: {state.get('cash', 0)}
Positions: {state.get('positions', [])}

News:
{news_context[:1500] if isinstance(news_context, str) else str(news_context)[:1500]}

Extra Context:
{extra_context[:1500] if isinstance(extra_context, str) else str(extra_context)[:1500]}
""".strip()

            try:
                past_reflections = memory.get_memories(situation_summary, MEMORY_TOP_K)
                print(f"[Memory] Retrieved {len(past_reflections)} reflections")
            except Exception as e:
                print(f"[Memory Error] {e}")
                past_reflections = []
        else:
            past_reflections = []

        # --------------------------------------------------------------
        # Step 6: Make LLM-powered decisions
        # --------------------------------------------------------------
        try:
            memory_context = "\n\n".join(past_reflections) if past_reflections else ""

            decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
                tok=tok,
                model=model,
                state=state,
                universe=WHITELIST,
                extra_context=extra_context,
                memory_context=memory_context
            )

            if isinstance(decisions, dict):
                decisions = [decisions]
            if not isinstance(decisions, list):
                decisions = []

            decisions = [d for d in decisions if isinstance(d, dict)]

            # keep only buy/sell decisions; if none, keep only one hold
            if decisions:
                active_decisions = [
                    d for d in decisions
                    if str(d.get("action", "")).lower() in ["buy", "sell"]
                ]

                if active_decisions:
                    decisions = active_decisions[:3]
                else:
                    hold_decisions = [
                        d for d in decisions
                        if str(d.get("action", "")).lower() == "hold"
                    ]

                    decisions = hold_decisions[:1] if hold_decisions else [{
                        "action": "hold",
                        "symbol": "AAPL",
                        "qty": 0,
                        "order_type": "market",
                        "limit_price": None,
                        "reason": "no clear edge"
                    }]

        except Exception as e:
            print(f"[Decision Error] {e}")
            decisions = []
            sft_prompt = ""
            sft_completion = ""
            tool_calls_made = 0

        total_tool_calls += tool_calls_made

        if not decisions:
            print("Warning: no valid decisions returned by llm_agent_decide")
            if loop_count < n_iterations:
                print(f"\nSleeping {LOOP_SECONDS} seconds before next loop...")
                time.sleep(LOOP_SECONDS)
            continue

        print(f"\n[Agent] Decisions ready | tool calls: {tool_calls_made}")

        loop_buys = 0
        loop_sells = 0
        loop_holds = 0

        for d in decisions:
            action = str(d.get("action", "hold")).lower().strip()
            symbol = d.get("symbol", "UNKNOWN")
            qty = d.get("qty", 0)
            reason = d.get("reason", "")

            print(f"  - {action.upper():5s} {symbol:5s} qty={qty} | {reason}")

            if action == "buy":
                loop_buys += 1
            elif action == "sell":
                loop_sells += 1
            else:
                loop_holds += 1

        total_buys += loop_buys
        total_sells += loop_sells
        total_holds += loop_holds

        print("\nDecision Summary:")
        print(f"Tool calls made: {tool_calls_made}")
        print(f"BUY: {loop_buys} | SELL: {loop_sells} | HOLD: {loop_holds}")

        # --------------------------------------------------------------
        # Step 6.5: Log to data collector
        # --------------------------------------------------------------
        if collector is not None:
            try:
                collector.log(
                    sft_prompt=sft_prompt,
                    sft_completion=sft_completion,
                    decisions=decisions,
                    equity=state.get("equity", 0.0),
                    tool_calls=tool_calls_made
                )

                if sft_prompt:
                    pending_outcomes.append({
                        "prompt": sft_prompt,
                        "loop": loop_count
                    })

            except Exception as e:
                print(f"[Collector Log Error] {e}")

        # --------------------------------------------------------------
        # Step 7: Execute orders
        # --------------------------------------------------------------
        real_trades_this_loop = 0

        for d in decisions:
            action = str(d.get("action", "hold")).lower().strip()

            if action in {"buy", "sell"}:
                try:
                    result = submit_order(tc, d)
                    print(f"[Order] {action.upper()} {d.get('symbol', 'UNKNOWN')} -> {result}")
                    real_trades_this_loop += 1

                    if memory is not None:
                        situation = f"""
                    State equity: {state.get('equity', 0)}
                    Cash: {state.get('cash', 0)}
                    Positions: {state.get('positions', [])}

                    News:
                    {news_context[:1500] if isinstance(news_context, str) else str(news_context)[:1500]}

                    Extra Context:
                    {extra_context[:1500] if isinstance(extra_context, str) else str(extra_context)[:1500]}

                    Past reflections:
                    {past_reflections[:3]}

                    Decision:
                    {d}
                    """.strip()

                        pending_reflections.append({
                            "loop": loop_count,
                            "situation": situation,
                            "decision": d,
                            "equity": state["equity"]
                        })

                except Exception as e:
                    print(f"[Order Error] {action.upper()} {d.get('symbol', 'UNKNOWN')} -> {e}")

            elif action == "hold":
                print(f"[HOLD] {d.get('symbol', 'UNKNOWN')} | {d.get('reason', 'No reason provided')}")

            else:
                print(f"[Unknown Action] {d}")

        total_real_trades += real_trades_this_loop

        print("\nLoop Execution Summary:")
        print(f"Real trades executed: {real_trades_this_loop}")
        print(f"Pending reflections now: {len(pending_reflections)}")
        print(f"Pending collector outcomes now: {len(pending_outcomes)}")

        if loop_count < n_iterations:
            print(f"\nSleeping {LOOP_SECONDS} seconds before next loop...")
            time.sleep(LOOP_SECONDS)

    print("\n" + "=" * 60)
    print("TRADING LOOP COMPLETE")
    print("=" * 60)
    print(f"Total tool calls: {total_tool_calls}")
    print(f"Total real trades: {total_real_trades}")
    print(f"Total BUY decisions: {total_buys}")
    print(f"Total SELL decisions: {total_sells}")
    print(f"Total HOLD decisions: {total_holds}")
    print(f"Pending reflections left: {len(pending_reflections)}")
    print(f"Pending collector outcomes left: {len(pending_outcomes)}")

### Run the Trading Loop

Use the **fine-tuned model** (`lora_model`) from Task 4 for live paper trading.

In [40]:
# Initialize memory system
try:
    memory = TradingMemory("trading_memory.json")
    print(f"Memory loaded: {len(memory.memories)} past entries")
except:
    memory = None
    print("Memory not available (implement Task 3 first)")

# Initialize data collector
collector = TradingDataCollector()

# Use fine-tuned model from Task 4 (fall back to base model if not available)
try:
    trading_model = lora_model
    print("Using fine-tuned LoRA model from Task 4")
except NameError:
    trading_model = model
    print("Warning: lora_model not found, using base model (run Task 4 first)")

print("\nStarting trading loop...")
print(f"  Symbols: {len(WHITELIST)} stocks")
print(f"  Iterations: 80")
print(f"  Interval: {LOOP_SECONDS}s")
print("-" * 60)

try:
    run_trading_loop(tc, tok, trading_model, memory=memory, collector=collector, n_iterations=80)
    print("\n\nTrading loop completed successfully!")
    print("\nData collected:")
    collector.summary()
except NotImplementedError:
    print("Task 5 not yet implemented")
except Exception as e:
    print(f"\nError: {e}")
    import traceback; traceback.print_exc()

Memory loaded: 0 past entries
Loaded 13 existing training entries
Using fine-tuned LoRA model from Task 4

Starting trading loop...
  Symbols: 100 stocks
  Iterations: 80
  Interval: 10s
------------------------------------------------------------

LOOP #1 / 80 | Pending reflections: 0

Current Account State:
Equity: $100,000.00
Cash:   $100,000.00
Positions: 0
[News] Loaded
[Extra Context] Loaded for ['TSLA', 'GOOG', 'AMZN']
[Memory] Retrieved 0 reflections
[Agent turn 1] → forced search_market_news({'topic': 'technology'})
[Agent] Finalising — 2 turn(s), 1 tool call(s)
[Agent] Finalising — 3 turn(s), 1 tool call(s)

[Agent] Decisions ready | tool calls: 1
  - BUY   MSFT  qty=1 | small starter position after research

Decision Summary:
Tool calls made: 1
BUY: 1 | SELL: 0 | HOLD: 0
[Order] BUY MSFT -> {'id': 'b20b2934-309f-4ff9-b5c3-7306fee576a0', 'status': 'OrderStatus.PENDING_NEW', 'symbol': 'MSFT', 'side': 'buy', 'qty': 1}

Loop Execution Summary:
Real trades executed: 1
Pending ref

### Optional: Inspect & Edit Memory

Use these cells to view, manually edit, download, or upload `trading_memory.json`.
You can add your own reflections or remove bad ones before the trading loop uses them.


In [ ]:
# =============================================================================
# OPTIONAL: Inspect & Edit trading_memory.json
# Run each sub-block independently as needed.
# =============================================================================

import os, json
from google.colab import files

memory_path = os.path.join(CACHE_DIR, "trading_memory.json")

# -- 1. View all memory entries --------------------------------------------
if os.path.exists(memory_path):
    with open(memory_path) as f:
        memories = json.load(f)
    print(f"Found {len(memories)} memory entries:\n")
    for i, m in enumerate(memories):
        label = m.get('label', '?')
        pnl   = m.get('pnl', '?')
        refl  = m.get('reflection', '')[:120].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      {refl}")
        print()
else:
    print("No trading_memory.json found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(memory_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment the block below after you have edited the file locally:
# uploaded = files.upload()   # select your edited trading_memory.json
# for fname, content in uploaded.items():
#     with open(memory_path, 'wb') as f:
#         f.write(content)
# with open(memory_path) as f:
#     memories = json.load(f)
# print(f"Reloaded {len(memories)} memories from uploaded file.")

# -- 4. Delete a bad entry by index ---------------------------------------
# Example: remove entry at index 2
# BAD_IDX = 2
# if os.path.exists(memory_path):
#     with open(memory_path) as f:
#         memories = json.load(f)
#     removed = memories.pop(BAD_IDX)
#     with open(memory_path, 'w') as f:
#         json.dump(memories, f, indent=2)
#     print(f"Removed entry {BAD_IDX}: {removed.get('reflection','')[:60]}")
#     print(f"{len(memories)} entries remain.")


---
## Bonus Challenges (Optional)

### 1. DPO (Direct Preference Optimization)
Instead of SFT (training only on "good" examples), use **DPO** to train on preference pairs:

```python
from trl import DPOTrainer, DPOConfig

# Format: (prompt, chosen_response, rejected_response)
# "chosen" = decisions labeled "good"
# "rejected" = decisions labeled "bad"

dpo_dataset = Dataset.from_dict({
    "prompt": [e["prompt"] for e in pairs],
    "chosen": [e["good_completion"] for e in pairs],
    "rejected": [e["bad_completion"] for e in pairs],
})

config = DPOConfig(
    output_dir="dpo_output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=5e-5,
)

trainer = DPOTrainer(
    model=lora_model,
    ref_model=model,       # frozen base model
    train_dataset=dpo_dataset,
    tokenizer=tok,
    args=config,
)
trainer.train()
```

**Key difference**: SFT teaches "do this", DPO teaches "prefer this over that" â€” generally produces better-calibrated models.

### 2. Vector Database RAG
Replace BM25 with semantic search:
- Use sentence embeddings (`sentence-transformers`)
- Store in FAISS vector index
- Retrieve by cosine similarity

### 3. Advanced Trading Strategy
- Position sizing based on confidence scores
- Stop-loss and take-profit logic
- Portfolio performance tracking with matplotlib

### 4. More Data Sources
- SEC filings (EDGAR API)
- Social media sentiment (Reddit, Twitter)
- Options chain data
- Earnings calendar integration